# Run Until...

In [ ]:
import os
# HF_TOKEN should be set via environment variables or loaded from .env
# Set before running: export HF_TOKEN="your-token"
if 'HF_TOKEN' not in os.environ:
    raise ValueError("HF_TOKEN must be set as an environment variable")

In [1]:
try:
    from rapidfuzz.fuzz import token_set_ratio, partial_ratio
    _HAS_RF = True
except ImportError:
    _HAS_RF = False
print(_HAS_RF)

True


In [2]:
# 🔧 0) Install training deps (run once per fresh notebook/kernel)
%uv pip install -U \
  transformers==4.41.2 \
  datasets==2.19.0 \
  peft==0.10.0 \
  accelerate==0.31.0 \
  sentencepiece==0.1.99 \
  einops==0.7.0 \
  safetensors==0.4.3

%uv pip install -U rapidfuzz==3.9.6

import os, torch
print("CUDA available:", torch.cuda.is_available())


Using Python 3.12.6 environment at: /usr/local
Resolved 64 packages in 106ms
Audited 64 packages in 0.10ms
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.6 environment at: /usr/local
Resolved 1 package in 36ms
Audited 1 package in 0.07ms
Note: you may need to restart the kernel to use updated packages.
CUDA available: True


# Here.

In [7]:
import json

def format_schema_for_prompt(table_code):
    """Given a table code (e.g., B02001), load and format schema for prompt prep."""
    path = f'/mnt/govquery-volume/schemas/{table_code.upper()}.json'
    with open(path, 'r') as f:
        schema = json.load(f)

    table_name = schema.get('table_name', table_code)
    columns = schema['columns']
    
    schema_lines = [f"### Table: {table_code.lower()} ({table_name})", "Columns:"]
    for col in columns:
        col_name = col['name']
        col_type = col['type'].lower()
        desc = col.get('description', '').strip()
        schema_lines.append(f"- {col_name} ({col_type}): {desc}")
    
    return "\n".join(schema_lines)

def build_schema_aware_prompt(schema_str, question):
    """
    Combines schema description with the natural language question
    into a full Text-to-SQL prompt input for T5-style models.
    """
    return f"{schema_str}\n\nQuestion: {question}\nSQL:"



In [8]:
schema_prompt = format_schema_for_prompt("B02001")
print(schema_prompt)

### Table: b02001 (Race)
Columns:
- geography_id (varchar): Geographic identifier
- year (integer): ACS 5‑year end year
- total_population (integer): Total population
- white_alone (integer): White alone
- black_or_african_american_alone (integer): Black or African‑American alone
- american_indian_alaska_native_alone (integer): American Indian and Alaska Native alone
- asian_alone (integer): Asian alone
- native_hawaiian_pacific_islander_alone (integer): Native Hawaiian and Other Pacific Islander alone
- some_other_race_alone (integer): Some other race alone
- two_or_more_races (integer): Two or more races
- total_population_moe (integer): MOE for total_population
- white_alone_moe (integer): MOE for white_alone
- black_or_african_american_alone_moe (integer): MOE for black_or_african_american_alone
- american_indian_alaska_native_alone_moe (integer): MOE for american_indian_alaska_native_alone
- asian_alone_moe (integer): MOE for asian_alone
- native_hawaiian_pacific_islander_alone_mo

In [9]:
test_question_1 = "How many people identified as Asian in Travis County, Texas in 2022?"
test_question_2 = "What is the margin of error for people who reported two or more races in Travis County in 2022?"
test_question_3 = "Show the number of Black or African-American individuals by county in Texas."

full_prompt = build_schema_aware_prompt(schema_prompt, test_question_1)
print(full_prompt)

### Table: b02001 (Race)
Columns:
- geography_id (varchar): Geographic identifier
- year (integer): ACS 5‑year end year
- total_population (integer): Total population
- white_alone (integer): White alone
- black_or_african_american_alone (integer): Black or African‑American alone
- american_indian_alaska_native_alone (integer): American Indian and Alaska Native alone
- asian_alone (integer): Asian alone
- native_hawaiian_pacific_islander_alone (integer): Native Hawaiian and Other Pacific Islander alone
- some_other_race_alone (integer): Some other race alone
- two_or_more_races (integer): Two or more races
- total_population_moe (integer): MOE for total_population
- white_alone_moe (integer): MOE for white_alone
- black_or_african_american_alone_moe (integer): MOE for black_or_african_american_alone
- american_indian_alaska_native_alone_moe (integer): MOE for american_indian_alaska_native_alone
- asian_alone_moe (integer): MOE for asian_alone
- native_hawaiian_pacific_islander_alone_mo

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import AutoModelForCausalLM
from huggingface_hub import login

login(token=os.environ['HF_TOKEN'])
# Load models and tokenizers
spider_model = AutoModelForSeq2SeqLM.from_pretrained("tscholak/t5.1.1.lm100k.base")
spider_tokenizer = AutoTokenizer.from_pretrained("tscholak/t5.1.1.lm100k.base")


lm100k_model = AutoModelForCausalLM.from_pretrained("defog/sqlcoder-7b-2")
lm100k_tokenizer = AutoTokenizer.from_pretrained("defog/sqlcoder-7b-2")



Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


model-00003-of-00003.safetensors:   0%|          | 0.00/3.59G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/515 [00:00<?, ?B/s]

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

def generate_sql(prompt):
    tokenizer = AutoTokenizer.from_pretrained("defog/sqlcoder-7b-2")
    model = AutoModelForCausalLM.from_pretrained("defog/sqlcoder-7b-2")
    model.to("cuda")  # L4 GPU

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=256)
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


def run_inference(model, tokenizer, prompt, max_tokens=256):
    pipe = pipeline("text2text-generation", model=model, tokenizer=tokenizer, device=0)
    result = pipe(prompt, max_length=max_tokens, truncation=True, do_sample=False)[0]["generated_text"]
    return result



In [6]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [7]:
# =========================
# 0) Imports & paths
# =========================
import os, gc, json, random, hashlib
from pathlib import Path
from datetime import datetime

import torch

SCHEMAS_DIR = Path("/mnt/govquery-volume/schemas/")
DATA_DIR    = Path("data"); DATA_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR  = Path("notebook/mnt/govquery-volume/models"); MODELS_DIR.mkdir(parents=True, exist_ok=True)

TABLES = [
    "B14001","B14002","B15003","B23025","B24010","B23006",
    "B19013","B19001","B20005","B01001","B02001","DP02"
]

# -------------------------
# Try to use the notebook's prompt builder; otherwise fallback.
# Expected signature: build_schema_aware_prompt(schema_dict, question) -> str
# -------------------------
if "build_schema_aware_prompt" not in globals():
    def build_schema_aware_prompt(schema_dict, question: str) -> str:
        # Minimal fallback based on the schema JSON structure you described
        code  = schema_dict.get("table_code", "UNKNOWN")
        name  = schema_dict.get("table_name", "")
        cols  = schema_dict.get("columns", [])
        col_lines = []
        for c in cols:
            col_lines.append(f"- {c['name']} ({c['type']}): {c.get('description','')}")
        col_block = "\n".join(col_lines[:200])  # keep prompt manageable
        return (
            f"### Table: {code} ({name})\n"
            f"Columns:\n{col_block}\n\n"
            f"Question: {question}\n"
            f"SQL:"
        )

# =========================
# 1) Utilities for schemas → dataset
# =========================
def sha256_path(p: Path) -> str:
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1<<20), b""):
            h.update(chunk)
    return h.hexdigest()

def parse_table_name_from_ddl(sql_ddl: str) -> str:
    # Expects something like: CREATE TABLE b01001 ( ...
    tokens = sql_ddl.strip().split()
    try:
        idx = [i for i,t in enumerate(tokens) if t.upper() == "TABLE"][0]
        return tokens[idx+1].strip("`\"")
    except Exception:
        # fallback to upper code
        return "unknown_table"

def load_schema(code: str) -> dict:
    with open(SCHEMAS_DIR/f"{code}.json", "r") as f:
        return json.load(f)

def column_list(schema: dict):
    return [c["name"] for c in schema["columns"]]

def make_safe_examples_for_table(schema: dict):
    """Create a few **safe** examples per table using the example row to avoid inventing geos/years."""
    examples = []
    example = schema.get("example_row", {})
    year    = example.get("year")
    geo     = example.get("geography_id")
    ddl     = schema.get("sql_ddl", "CREATE TABLE unknown (id INT)")
    table   = parse_table_name_from_ddl(ddl)
    cols    = column_list(schema)

    # numeric non-MOE columns
    numeric_cols = [c["name"] if isinstance(c, dict) else c for c in schema["columns"]
                    if (isinstance(c, dict) and c["name"] not in ("geography_id","year") and not c["name"].endswith("_moe"))]
    numeric_cols = [c for c in numeric_cols if isinstance(c, str)]

    # MOE columns
    moe_cols = [c["name"] for c in schema["columns"] if isinstance(c, dict) and c["name"].endswith("_moe")]

    # 1) value lookup
    if numeric_cols and geo and year:
        col = numeric_cols[0]
        q   = f"Show the value of {col.replace('_',' ')} for this geography in {year}."
        prompt = build_schema_aware_prompt(schema, q)
        sql    = f"SELECT {col} FROM {table} WHERE geography_id = '{geo}' AND year = {year};"
        examples.append({"table_code": schema.get("table_code"), "table_name": schema.get("table_name"),
                         "prompt": prompt, "sql": sql})

    # 2) MOE lookup (if exists)
    if moe_cols and geo and year:
        moe = moe_cols[0]
        base = moe[:-4].replace('_',' ')
        q   = f"What is the margin of error for {base} in {year}?"
        prompt = build_schema_aware_prompt(schema, q)
        sql    = f"SELECT {moe} FROM {table} WHERE geography_id = '{geo}' AND year = {year};"
        examples.append({"table_code": schema.get("table_code"), "table_name": schema.get("table_name"),
                         "prompt": prompt, "sql": sql})

    # 3) Another numeric column if available
    if len(numeric_cols) > 1 and geo and year:
        col = numeric_cols[-1]
        q   = f"Return {col.replace('_',' ')} for this geography in {year}."
        prompt = build_schema_aware_prompt(schema, q)
        sql    = f"SELECT {col} FROM {table} WHERE geography_id = '{geo}' AND year = {year};"
        examples.append({"table_code": schema.get("table_code"), "table_name": schema.get("table_name"),
                         "prompt": prompt, "sql": sql})

    return examples

# Build dataset
items = []
schema_fingerprint = {}
for code in TABLES:
    spath = SCHEMAS_DIR/f"{code}.json"
    s = load_schema(code)
    schema_fingerprint[code] = sha256_path(spath)
    items.extend(make_safe_examples_for_table(s))

random.shuffle(items)
split = int(0.9 * len(items))
train_items, val_items = items[:split], items[split:]

DATA_DIR.mkdir(exist_ok=True, parents=True)
with open(DATA_DIR/"train.jsonl", "w") as f:
    for ex in train_items: f.write(json.dumps(ex)+"\n")
with open(DATA_DIR/"val.jsonl", "w") as f:
    for ex in val_items: f.write(json.dumps(ex)+"\n")

print(f"Built dataset: {len(train_items)} train / {len(val_items)} val  (total {len(items)})")
print("Schema files fingerprinted:", len(schema_fingerprint))

# Save fingerprints to both model homes for provenance
(MODELS_DIR/"t5_lora").mkdir(parents=True, exist_ok=True)
(MODELS_DIR/"sqlcoder_qlora").mkdir(parents=True, exist_ok=True)
with open(MODELS_DIR/"t5_lora/schema_fingerprint.json","w") as f: json.dump(schema_fingerprint, f, indent=2)
with open(MODELS_DIR/"sqlcoder_qlora/schema_fingerprint.json","w") as f: json.dump(schema_fingerprint, f, indent=2)


Built dataset: 31 train / 4 val  (total 35)
Schema files fingerprinted: 12


# Sanity Check to Remove Incompatibile DP | Bitsandbyte

In [8]:
# 🔧 remove the conflicting package so Trainer can import
%uv pip uninstall bitsandbytes

# (optional: make sure it's gone in this process)
import sys
sys.modules.pop("bitsandbytes", None)


Using Python 3.12.6 environment at: /usr/local
Uninstalled 1 package in 8ms
 - bitsandbytes==0.43.1
Note: you may need to restart the kernel to use updated packages.


# Training Models | T5 + SQLCoder Lora

In [9]:
# =========================
# 2) T5 (tscholak/t5.1.1.lm100k.base) → LoRA fine-tune
# =========================
from datasets import Dataset as HFDataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq, Trainer, TrainingArguments
)
from peft import LoraConfig, get_peft_model

# Load
t5_name = "tscholak/t5.1.1.lm100k.base"
t5_tok  = AutoTokenizer.from_pretrained(t5_name, use_fast=True)
t5_base = AutoModelForSeq2SeqLM.from_pretrained(t5_name)

# LoRA adapter
t5_lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q","v","k","o","wi","wo"],  # reasonable defaults for T5
    task_type="SEQ_2_SEQ_LM"
)
t5_model = get_peft_model(t5_base, t5_lora_cfg)

# Tokenization fn
def t5_tok_fn(batch):
    model_inputs = t5_tok(batch["prompt"], truncation=True, padding=True, max_length=1536)
    labels = t5_tok(text_target=batch["sql"], truncation=True, padding=True, max_length=512)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

t5_train = HFDataset.from_json(str(DATA_DIR/"train.jsonl")).map(t5_tok_fn, batched=True, remove_columns=["prompt","sql","table_code","table_name"])
t5_val   = HFDataset.from_json(str(DATA_DIR/"val.jsonl")).map(t5_tok_fn,   batched=True, remove_columns=["prompt","sql","table_code","table_name"])

# Training
t5_args = TrainingArguments(
    output_dir=str(MODELS_DIR/"t5_lora/checkpoints"),
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    num_train_epochs=5,
    evaluation_strategy="steps",
    eval_steps=200,
    save_steps=200,
    logging_steps=50,
    load_best_model_at_end=True,
    bf16=torch.cuda.is_available(),
    report_to=[],
)

t5_trainer = Trainer(
    model=t5_model,
    args=t5_args,
    train_dataset=t5_train,
    eval_dataset=t5_val,
    data_collator=DataCollatorForSeq2Seq(t5_tok, model=t5_model),
)

t5_trainer.train()

# Save adapter + tokenizer + generation config
T5_OUT = MODELS_DIR/"t5_lora"
t5_model.save_pretrained(str(T5_OUT))
t5_tok.save_pretrained(str(T5_OUT))

# Generation config (stop at ';' via serving logic; EOS remains standard)
with open(T5_OUT/"generation_config.json","w") as f:
    json.dump({
        "max_new_tokens": 256,
        "do_sample": False,
        "num_beams": 1,
        "early_stopping": True
    }, f, indent=2)

# Cleanup GPU
del t5_trainer, t5_model, t5_base
gc.collect(); torch.cuda.empty_cache()
print("Saved T5 LoRA to", T5_OUT)


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/31 [00:00<?, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Step,Training Loss,Validation Loss


/usr/local/lib/python3.12/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Saved T5 LoRA to notebook/mnt/govquery-volume/models/t5_lora


# T5 Sanity Check #1

In [10]:
# --- T5 sanity inference on one real question (fixed) ---
import json, torch, gc
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

SCHEMAS_DIR = Path("/mnt/govquery-volume/schemas/")
T5_OUT = Path("/mnt/govquery-volume/models/t5_lora")

code = "B02001"  # Race
with open(SCHEMAS_DIR/f"{code}.json") as f:
    schema = json.load(f)

question = "How many people identified as Asian in Travis County, Texas in 2022?"
prompt = build_schema_aware_prompt(schema, question)

tok = AutoTokenizer.from_pretrained(str(T5_OUT))
mdl = AutoModelForSeq2SeqLM.from_pretrained(str(T5_OUT)).to("cuda" if torch.cuda.is_available() else "cpu")

# 🔧 Truncate the long schema-aware prompt to model’s input window
enc = tok(prompt, return_tensors="pt", truncation=True, padding=True, max_length=1536).to(mdl.device)

out = mdl.generate(**enc, max_new_tokens=256, do_sample=False, num_beams=1)
sql_t5 = tok.decode(out[0], skip_special_tokens=True).strip()
sql_t5 = (sql_t5.split(";")[0] + ";") if ";" in sql_t5 else sql_t5
print("T5 SQL:\n", sql_t5)

del mdl; gc.collect(); torch.cuda.empty_cache()

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


T5 SQL:
 - asian_alone_moe (INTEGER): Asian alone - black_or_african_american_alone (INTEGER): Black or AfricanAmerican alone - native_hawaiian_pacific_islander_alone_moe (INTEGER): Native Hawaiian and Other Pacific Islander alone - some_other_race_alone_moe (INTEGER): MOE for some_other_race_alone - some_other_race_alone_moe (INTEGER): MOE for some_other_race_alone - some_other_race_alone_moe (INTEGER): MOE for some_other_race_alone - some_other_race_alone_moe (INTEGER): MOE for some_other_race_alone - some_other_race_alone_moe (INTEGER): MOE for some_other_race_alone - some_other_race_alone_moe (INTEGER): MOE for some_


# SQLCoder 7B Training

In [11]:
# --- SQLCoder-7B-2 LoRA (no bitsandbytes) — fixed pad token ---
import os, json, torch, gc
from pathlib import Path
from datasets import Dataset as HFDataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    Trainer, TrainingArguments, DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model

assert os.environ.get("HF_TOKEN"), "Set os.environ['HF_TOKEN'] = 'hf_xxx'"
hf_token = os.environ["HF_TOKEN"]

DATA_DIR   = Path("data")
MODELS_DIR = Path("/mnt/govquery-volume/models/"); MODELS_DIR.mkdir(parents=True, exist_ok=True)
SQL_OUT    = MODELS_DIR/"sqlcoder_lora_fp"; SQL_OUT.mkdir(parents=True, exist_ok=True)

model_name = "defog/sqlcoder-7b-2"
tok  = AutoTokenizer.from_pretrained(model_name, use_fast=True, token=hf_token)

# 🔧 pad-token + padding side
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "right"

base = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    token=hf_token,
)
# keep model config consistent with tokenizer
base.config.pad_token_id = tok.pad_token_id

# LoRA adapter config
lora_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM")
model = get_peft_model(base, lora_cfg)

def pack(batch):
    texts = [p + s for p, s in zip(batch["prompt"], batch["sql"])]
    out = tok(texts, truncation=True, padding=True, max_length=2048)
    # causal LM learns next-token prediction over full sequence
    out["labels"] = out["input_ids"]
    return out

train = HFDataset.from_json(str(DATA_DIR/"train.jsonl")).map(
    pack, batched=True, remove_columns=["prompt","sql","table_code","table_name"]
)
val = HFDataset.from_json(str(DATA_DIR/"val.jsonl")).map(
    pack, batched=True, remove_columns=["prompt","sql","table_code","table_name"]
)

args = TrainingArguments(
    output_dir=str(SQL_OUT/"checkpoints"),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-4,
    num_train_epochs=3,
    evaluation_strategy="steps",
    eval_steps=200,
    save_steps=200,
    logging_steps=50,
    bf16=torch.cuda.is_available(),
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train,
    eval_dataset=val,
    data_collator=DataCollatorForLanguageModeling(tok, mlm=False),
)

trainer.train()

model.save_pretrained(str(SQL_OUT))
tok.save_pretrained(str(SQL_OUT))
with open(SQL_OUT/"generation_config.json","w") as f:
    json.dump({"max_new_tokens":256,"do_sample":False,"num_beams":1,"early_stopping":True,"stop_strings":[";"]}, f, indent=2)

del trainer, model, base
gc.collect(); torch.cuda.empty_cache()
print("Saved SQLCoder LoRA (FP) →", SQL_OUT)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Map:   0%|          | 0/31 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Step,Training Loss,Validation Loss


/usr/local/lib/python3.12/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Saved SQLCoder LoRA (FP) → /mnt/govquery-volume/models/sqlcoder_lora_fp


# Test Batter for 12 JSON Files

In [12]:
# --- Build a compact test battery from /schemas for all 12 tables ---
import json, random
from pathlib import Path

SCHEMAS_DIR = Path("/mnt/govquery-volume/schemas")
OUT_DIR = Path("/mnt/govquery-volume/outputs"); OUT_DIR.mkdir(parents=True, exist_ok=True)

TABLES = ["B14001","B14002","B15003","B23025","B24010","B23006",
          "B19013","B19001","B20005","B01001","B02001","DP02"]

def load_schema(code):
    with open(SCHEMAS_DIR/f"{code}.json") as f: return json.load(f)

def pick_cols(schema):
    cols = [c["name"] for c in schema["columns"] if c["name"] not in ("geography_id","year")]
    base  = [c for c in cols if not c.endswith("_moe")]
    moes  = [c for c in cols if c.endswith("_moe")]
    return base[:3], moes[:1]  # keep test short & fast

tests = []
for code in TABLES:
    s = load_schema(code)
    example_geo  = s["example_row"]["geography_id"]
    example_year = s["example_row"]["year"]
    base_cols, moe_cols = pick_cols(s)

    if base_cols:
        col = base_cols[0]
        q = f"Return {col.replace('_',' ')} for this geography in {example_year}."
        tests.append({"table": code, "question": q})

    if moe_cols:
        c = moe_cols[0][:-4].replace("_"," ")
        q = f"What is the margin of error for {c} in {example_year}?"
        tests.append({"table": code, "question": q})

    if len(base_cols) > 1:
        col = base_cols[-1]
        q = f"Show the value of {col.replace('_',' ')} for this geography in {example_year}."
        tests.append({"table": code, "question": q})

random.shuffle(tests)
with open(OUT_DIR/"tests.json","w") as f: json.dump(tests, f, indent=2)
print(f"Wrote {len(tests)} tests → {OUT_DIR/'tests.json'}")


Wrote 35 tests → /mnt/govquery-volume/outputs/tests.json


# A/B Testing

In [21]:
# ab_infer_hardened.py
# Robust A/B inference → JSONL (SQL-only, geo/year enforced, MOE/value aligned)
#
# ENHANCED SYNONYM RECOGNITION FEATURES:
# ======================================
#
# 1. OPTIONAL RAPIDFUZZ INTEGRATION
#    - Automatically detects if rapidfuzz is available
#    - Uses hybrid scoring: 50% token overlap + 50% RapidFuzz similarity
#    - Graceful fallback when RapidFuzz not installed
#
# 2. ALIAS NORMALIZATION
#    - Expands common synonyms: "women"→"female", "pre-k"→"nursery preschool"
#    - Handles age phrases: "16+"→"16plus", "16 and over"→"16plus"
#    - MOE synonyms: "error margin"→"margin of error", "±"→"margin of error"
#    - Race/ethnicity: "black alone"→"black or african american alone"
#
# 3. CONFIGURABLE THRESHOLDS
#    - SELECTION_THRESHOLD: Minimum score to select a column (default: 0.62)
#    - TIE_DELTA: Minimum difference to break ties (default: 0.02)
#    - Adjust these constants at top of file to tune performance
#
# 4. SYNONYM TEST SUITE
#    - Run with: python ab_infer_hardened.py --test-synonyms
#    - Or set environment: RUN_SYNONYM_TESTS=1
#    - Tests 30+ synonym cases across hot tables (B01001, B02001, B19001, etc.)
#    - Validates alias expansion, MOE detection, and column selection
#    - No external I/O - uses mock schemas for testing
#
# USAGE EXAMPLES:
# ==============
# Normal execution: python ab_infer_hardened.py
# Run synonym tests: python ab_infer_hardened.py --test-synonyms
# Tune thresholds: Edit SELECTION_THRESHOLD and TIE_DELTA at top of file
#
# GUARDRAILS (unchanged):
# ======================
# - Exactly one SELECT ... ; per output
# - Correct table name from schema
# - WHERE geography_id = ... AND year = 2022 enforced
# - MOE intent respected: MOE phrasing → *_moe columns
# - Deterministic behavior, no new I/O, no device/path changes

import json, gc, re, torch
from pathlib import Path
from typing import Optional, Dict, Set, List, Tuple, Union, Any
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
    LogitsProcessorList,
    NoBadWordsLogitsProcessor,
)

# Optional RapidFuzz dependency for enhanced synonym matching
try:
    from rapidfuzz.fuzz import token_set_ratio, partial_ratio
    _HAS_RF = True
except ImportError:
    _HAS_RF = False

# ----------------------------- Configurable Thresholds -----------------------------
# Tune these thresholds based on synonym test results
SELECTION_THRESHOLD = 0.62  # Minimum score to select a column (0.0-1.0)
TIE_DELTA = 0.02           # Minimum difference to break ties (0.0-1.0)

# ----------------------------- Paths -----------------------------
SCHEMAS_DIR = Path("/mnt/govquery-volume/schemas")
OUT_DIR     = Path("/mnt/govquery-volume/models/outputs")
T5_DIR      = Path("/mnt/govquery-volume/models/t5_lora")
SQL_DIR     = Path("/mnt/govquery-volume/models/sqlcoder_lora_fp")
TESTS_PATH  = OUT_DIR / "tests.json"
OUT_PATH    = OUT_DIR / "ab_compare_all_postgeo.jsonl"

# ----------------------------- IO -----------------------------
with open(TESTS_PATH) as f:
    tests = json.load(f)

def load_schema(code: str) -> Dict[str, Any]:
    """Load schema for a given table code."""
    with open(SCHEMAS_DIR / f"{code}.json") as f:
        return json.load(f)

# ----------------------------- Prompting -----------------------------
def build_schema_aware_prompt(schema: Dict[str, Any], question: str) -> str:
    """
    Build a schema-aware prompt for the model.
    
    Args:
        schema: Table schema dictionary
        question: Natural language question
        
    Returns:
        Formatted prompt string
    """
    table_code = (schema.get("table_code") or schema.get("table") or "UNKNOWN").lower()
    cols = [c.get("name") for c in schema.get("columns", [])]
    ddlish = f"Table: {table_code}\nColumns: {', '.join(cols)}"
    return (
        "You are a SQL expert for ACS-like star-schema tables.\n"
        "Write a single SELECT statement answering the question using only the provided table/columns.\n\n"
        f"{ddlish}\n\n"
        f"Question: {question}\n"
    )

FORMAT_CONTRACT = (
    "\nOutput requirements:\n"
    "- Return exactly ONE SQL query starting with SELECT and ending with a semicolon.\n"
    "- Do NOT include JSON, DDL, comments, or explanations.\n"
    "- Use only columns present in the schema.\n"
)

# ----------------------------- Text utils -----------------------------
def truncate_semicolon(txt: str) -> str:
    """Ensure text ends with a semicolon."""
    return (txt.split(';')[0] + ';') if ';' in txt else txt.strip() + ';'

# ----------------------------- Enhanced normalization and alias expansion -----------------------------
# Alias expansion mappings for better synonym recognition
ALIAS_EXPANSIONS = {
    # Gender aliases
    "women": "female",
    "men": "male",
    "woman": "female", 
    "man": "male",
    
    # Race/ethnicity aliases
    "black alone": "black or african american alone",
    "african american": "black or african american alone",
    "latino": "hispanic or latino",
    "hispanic": "hispanic or latino",
    
    # Education aliases
    "pre-k": "nursery preschool",
    "prekindergarten": "nursery preschool",
    "pre school": "nursery preschool",
    "hs grad": "high school graduate or higher",
    "hs graduate": "high school graduate or higher",
    "high school grad": "high school graduate or higher",
    "high school graduate": "high school graduate or higher",
    "college grad": "college graduate",
    "bachelor": "bachelors degree",
    "bachelors": "bachelors degree",
    "masters": "graduate degree",
    "phd": "graduate degree",
    "doctorate": "graduate degree",
    
    # Age aliases
    "16+": "16plus",
    "16 and older": "16plus", 
    "16 years and over": "16plus",
    "16 and over": "16plus",
    "25+": "25plus",
    "25 and older": "25plus",
    "25 years and over": "25plus",
    "25 and over": "25plus",
    "3+": "3plus",
    "3 and older": "3plus",
    "3 years and over": "3plus",
    "3 and over": "3plus",
    
    # MOE aliases
    "error margin": "margin of error",
    "uncertainty": "margin of error",
    "±": "margin of error",
    "ci": "confidence interval",
    "confidence interval": "margin of error",
    "standard error": "margin of error",
    "se": "margin of error",
    
    # Employment aliases
    "jobless": "unemployed",
    "out of work": "unemployed",
    "not working": "unemployed",
    "labor force": "civilian labor force",
    
    # Housing aliases
    "own": "owner occupied",
    "rent": "renter occupied",
    "renting": "renter occupied",
    "home value": "median home value",
    "rental": "median rent",
}

def normalize_text(text: str) -> str:
    """Normalize text for better matching: lowercase, trim, collapse whitespace, handle underscores."""
    if not text:
        return ""
    
    # Basic normalization
    normalized = text.lower().strip()
    
    # Collapse multiple whitespace
    normalized = re.sub(r'\s+', ' ', normalized)
    
    # Replace underscores with spaces for better matching
    normalized = normalized.replace('_', ' ')
    
    # Remove punctuation except for specific cases
    normalized = re.sub(r'[^\w\s\-\+]', ' ', normalized)
    
    # Collapse whitespace again
    normalized = re.sub(r'\s+', ' ', normalized).strip()
    
    return normalized

def expand_aliases(text: str) -> str:
    """Expand common aliases to canonical forms."""
    normalized = normalize_text(text)
    
    # Apply alias expansions
    for alias, canonical in ALIAS_EXPANSIONS.items():
        # Use word boundaries for better matching
        pattern = r'\b' + re.escape(alias) + r'\b'
        if re.search(pattern, normalized):
            normalized = re.sub(pattern, canonical, normalized)
    
    return normalized

def extract_tokens(text: str) -> Set[str]:
    """Extract meaningful tokens from text."""
    expanded = expand_aliases(text)
    # Extract alphanumeric tokens
    tokens = set(re.findall(r'[a-z0-9]+', expanded))
    # Filter out very short tokens that are likely noise
    return {t for t in tokens if len(t) >= 2}

# ----------------------------- Extract + Validate -----------------------------
# Pre-compiled regex patterns for performance
SELECT_RE = re.compile(r"(?is)\bselect\b.+?\sfrom\s+([a-z_][\w]*)\b.*?;", re.DOTALL)
SELECT_BASIC_RE = re.compile(r"(?is)\bselect\b.+?;", re.DOTALL)
MARKDOWN_FENCE_RE = re.compile(r"(?is)^```+[a-z]*\s*\n?(.*?)\n?```+", re.DOTALL)
JSON_SQL_RE = re.compile(r"(?is)['\"]sql['\"]\s*:\s*['\"](.*?)['\"]", re.DOTALL)
PREFIX_CLEANUP_RE = re.compile(r"(?is)^```+sql|^```+|^answer\s*:|^sql\s*:|^query\s*:")

def extract_sql(generated_text: str, prompt: str) -> str:
    """Extract SQL query from model output with robust fallback handling."""
    txt = generated_text.strip()
    
    # Remove prompt if present
    if txt.startswith(prompt):
        txt = txt[len(prompt):].strip()
    
    # Remove common prefixes and markdown
    txt = PREFIX_CLEANUP_RE.sub("", txt).strip()
    
    # Try to extract from markdown fences
    markdown_match = MARKDOWN_FENCE_RE.search(txt)
    if markdown_match:
        txt = markdown_match.group(1).strip()
    
    # Try to extract from JSON-like structures
    json_match = JSON_SQL_RE.search(txt)
    if json_match:
        txt = json_match.group(1).strip()
    
    # Look for complete SELECT statements
    m = SELECT_RE.search(txt)
    if m:
        return m.group(0).strip()
    
    # Fallback to basic SELECT pattern
    basic_match = SELECT_BASIC_RE.search(txt)
    if basic_match:
        return truncate_semicolon(basic_match.group(0).strip())
    
    # Last resort: return truncated text with semicolon
    return truncate_semicolon(txt)

def is_valid_sql(sql: str, table_code: str) -> bool:
    """Check if SQL is valid and references the correct table."""
    return bool(SELECT_RE.search(sql)) and (table_code.lower() in sql.lower())

# ----------------------------- Fallback builder -----------------------------
# Pre-compiled regex for column selection
COLUMN_CLEANUP_RE = re.compile(r"[^a-z0-9 ]")

def choose_column(schema: Dict[str, Any], question: str) -> str:
    """Choose the best column for a question using enhanced fuzzy matching."""
    # Use enhanced token extraction
    q_tokens = extract_tokens(question)
    best, best_score = None, -1
    
    for col in schema.get("columns", []):
        name = col.get("name", "")
        description = col.get("description", "")
        
        # Enhanced matching using normalized text
        col_text = normalize_text(name + " " + description)
        col_tokens = extract_tokens(col_text)
        
        # Calculate overlap score
        score = len(q_tokens & col_tokens)
        
        # Boost important columns
        if any(key in col_text for key in ["total", "moe", "median"]) and any(w in q_tokens for w in ["total", "moe", "median"]):
            score += 2
            
        if score > best_score:
            best, best_score = name, score
    
    return best or schema.get("columns", [{}])[0].get("name", "*")

def build_output_sql(schema: Dict[str, Any], question: str, ctx: Dict[str, Any] = None, year: int = 2022) -> str:
    """Build a fallback SQL query for a given schema and question with concrete geography."""
    tbl = (schema.get("table_code") or schema.get("table") or schema.get("table_name", "").split()[0] or "unknown").lower()
    col = choose_column(schema, question)
    
    # Use comprehensive geography inference
    if ctx:
        target_geo, _, _, _ = infer_geo_from_ctx(ctx, question)
        is_multi_geo = isinstance(target_geo, list)
        if is_multi_geo:
            target_geo = [normalize_geo_for_table(geo, schema)[0] for geo in target_geo]
            geo_list = "','".join(target_geo)
            geo_predicate = f"geography_id IN ('{geo_list}')"
            select_cols = f"geography_id, {col}"
        else:
            target_geo = normalize_geo_for_table(target_geo, schema)[0]
            geo_predicate = f"geography_id = '{target_geo}'"
            select_cols = col
    else:
        geo_predicate = "geography_id = '01000US'"  # National fallback
        select_cols = col
    
    return f"SELECT {select_cols} FROM {tbl} WHERE {geo_predicate} AND year = {year};"

# ----------------------------- MOE vs Value post-fixer -----------------------------
SELECT_COL_RE = re.compile(r"(?is)^\s*select\s+(.+?)\s+from\s+([a-z_][\w]*)\s+where\b")
WORD_EXTRACT_RE = re.compile(r"[a-z0-9]+")

# Comprehensive MOE detection patterns
MOE_HINTS: Set[str] = {
    "moe", "margin of error", "margin-of-error", "margin", "error",
    "uncertainty", "confidence interval", "standard error", "se",
    "plus or minus", "±", "plus-minus", "error range", "error bound",
    "error margin", "ci", "confidence", "interval", "bound", "range"
}

# Patterns that indicate value/estimate requests (not MOE)
VALUE_HINTS: Set[str] = {
    "value", "estimate", "number", "count", "total", "amount",
    "how many", "what is", "return", "show", "give", "find",
    "population", "households", "income", "percentage", "percent"
}

def wants_moe(question: str) -> bool:
    """Determine if question is asking for margin of error vs actual value."""
    # Use enhanced normalization and alias expansion
    q = expand_aliases(question)
    
    # Check for explicit MOE indicators
    moe_score = sum(1 for hint in MOE_HINTS if hint in q)
    
    # Check for value indicators (these reduce MOE likelihood)
    value_score = sum(1 for hint in VALUE_HINTS if hint in q)
    
    # Special patterns that strongly indicate MOE
    strong_moe_patterns = [
        r"\bmargin\s+of\s+error\b",
        r"\berror\s+for\b",
        r"\buncertainty\s+in\b",
        r"\bconfidence\s+interval\b",
        r"\bstandard\s+error\b",
        r"\bplus\s+or\s+minus\b",
        r"\b±\b"
    ]
    
    for pattern in strong_moe_patterns:
        if re.search(pattern, q):
            return True
    
    # If MOE hints outweigh value hints, likely wants MOE
    return moe_score > value_score

def normalize(col: str) -> str:
    """Normalize column name by removing quotes and whitespace."""
    return col.strip().strip("`\"")

# ----------------------------- RapidFuzz scoring -----------------------------
def rapidfuzz_score(question: str, column_name: str, column_description: str = "") -> float:
    """Calculate RapidFuzz similarity score between question and column."""
    if not _HAS_RF:
        return 0.0
    
    # Normalize inputs
    q_norm = normalize_text(question)
    name_norm = normalize_text(column_name)
    desc_norm = normalize_text(column_description)
    
    # Create candidate strings for comparison
    candidates = [
        name_norm,
        name_norm + " " + desc_norm if desc_norm else name_norm,
        desc_norm if desc_norm else ""
    ]
    
    best_score = 0.0
    for candidate in candidates:
        if not candidate:
            continue
            
        # Use hybrid scoring: 60% token_set_ratio + 40% partial_ratio
        token_score = token_set_ratio(q_norm, candidate)
        partial_score = partial_ratio(q_norm, candidate)
        hybrid_score = 0.6 * token_score + 0.4 * partial_score
        
        best_score = max(best_score, hybrid_score)
    
    # Normalize to 0-1 range
    return best_score / 100.0

# Pre-compiled KEY_MAP patterns for performance
KEY_MAP_PATTERNS = [
    # Income and economic indicators
    (re.compile(r"\bmedian household income\b", re.I), "median_household_income"),
    (re.compile(r"\bhh income 10k to 14999\b", re.I), "hh_income_10k_to_14999"),
    (re.compile(r"\bhousehold income\b", re.I), "household_income"),
    (re.compile(r"\bper capita income\b", re.I), "per_capita_income"),
    
    # Population demographics
    (re.compile(r"\btotal population 16(\s|_)over\b|\bpopulation 16\b", re.I), "total_population_16_over"),
    (re.compile(r"\btotal population 25( |_)over\b", re.I), "total_population_25_over"),
    (re.compile(r"\btotal population 3\+|\btotal population 3plus\b", re.I), "total_population_3plus"),
    (re.compile(r"\btotal population\b", re.I), "total_population"),
    (re.compile(r"\bcivilian population 18( |_)over\b", re.I), "civilian_population_18_over"),
    (re.compile(r"\bworking age population\b", re.I), "working_age_population"),
    
    # Households and families
    (re.compile(r"\btotal households\b", re.I), "total_households"),
    (re.compile(r"\bfamily households\b", re.I), "family_households"),
    (re.compile(r"\bnonfamily households\b", re.I), "nonfamily_households"),
    
    # Race and ethnicity
    (re.compile(r"\bblack or african american alone\b", re.I), "black_or_african_american_alone"),
    (re.compile(r"\bwhite alone\b", re.I), "white_alone"),
    (re.compile(r"\bhispanic or latino\b", re.I), "hispanic_or_latino"),
    (re.compile(r"\basian alone\b", re.I), "asian_alone"),
    (re.compile(r"\bnative american\b", re.I), "native_american"),
    
    # Gender demographics
    (re.compile(r"\bmale total\b", re.I), "male_total"),
    (re.compile(r"\bfemale population\b", re.I), "female_population"),
    (re.compile(r"\bmale not enrolled\b", re.I), "male_not_enrolled"),
    (re.compile(r"\bfemale not enrolled\b", re.I), "female_not_enrolled"),
    (re.compile(r"\bmale service\b", re.I), "male_service"),
    (re.compile(r"\bfemale service\b", re.I), "female_service"),
    
    # Education
    (re.compile(r"\bnursery (pre)?school\b", re.I), "nursery_preschool"),
    (re.compile(r"\bhigh school graduate or higher\b", re.I), "high_school_graduate_or_higher"),
    (re.compile(r"\bged\b|\bged or equivalent\b", re.I), "ged_or_equivalent"),
    (re.compile(r"\bless than high school total\b", re.I), "less_than_high_school_total"),
    (re.compile(r"\bless than high school unemployed\b", re.I), "less_than_high_school_unemployed"),
    (re.compile(r"\bbachelor's degree\b", re.I), "bachelors_degree"),
    (re.compile(r"\bgraduate degree\b", re.I), "graduate_degree"),
    (re.compile(r"\bcollege graduate\b", re.I), "college_graduate"),
    
    # Employment and labor
    (re.compile(r"\bcivilian labor force\b", re.I), "civilian_labor_force"),
    (re.compile(r"\bunemployed\b", re.I), "unemployed"),
    (re.compile(r"\bemployed\b", re.I), "employed"),
    (re.compile(r"\bnot in labor force\b", re.I), "not_in_labor_force"),
    (re.compile(r"\bunemployment rate\b", re.I), "unemployment_rate"),
    
    # Housing
    (re.compile(r"\bowner occupied\b", re.I), "owner_occupied"),
    (re.compile(r"\brenter occupied\b", re.I), "renter_occupied"),
    (re.compile(r"\bmedian home value\b", re.I), "median_home_value"),
    (re.compile(r"\bmedian rent\b", re.I), "median_rent"),
    
    # Age groups
    (re.compile(r"\bunder 18\b", re.I), "under_18"),
    (re.compile(r"\b18 to 64\b", re.I), "18_to_64"),
    (re.compile(r"\b65 and over\b", re.I), "65_and_over"),
    (re.compile(r"\bmedian age\b", re.I), "median_age"),
]

def best_column_for_question(schema: Dict[str, Any], question: str) -> Optional[str]:
    """Find the best column for a question using hybrid scoring (KEY_MAP + token overlap + RapidFuzz)."""
    want_moe = wants_moe(question)
    cols = [c["name"] for c in schema.get("columns", [])]
    cols_set = set(cols)
    
    # First, try exact pattern matching with pre-compiled patterns
    for pat, base in KEY_MAP_PATTERNS:
        if pat.search(question):
            # Try MOE version first if that's what's wanted
            if want_moe:
                moe_cand = base + "_moe"
                if moe_cand in cols_set:
                    return moe_cand
            # Try base version
            if base in cols_set:
                return base
            # If base not found, try MOE version as fallback
            if not want_moe:
                moe_cand = base + "_moe"
                if moe_cand in cols_set:
                    return moe_cand
    
    # Hybrid scoring: token overlap + RapidFuzz
    q_tokens = extract_tokens(question)
    best_col = None
    best_score = -1.0
    
    for col in schema.get("columns", []):
        name = col.get("name", "")
        description = col.get("description", "")
        
        # Token overlap score (0-1)
        col_text = normalize_text(name + " " + description)
        col_tokens = extract_tokens(col_text)
        token_overlap = len(q_tokens & col_tokens) / max(len(q_tokens), 1)
        
        # RapidFuzz score (0-1)
        rf_score = rapidfuzz_score(question, name, description)
        
        # Hybrid score: 50% token overlap + 50% RapidFuzz
        final_score = 0.5 * token_overlap + 0.5 * rf_score
        
        # MOE alignment bonus/penalty
        if want_moe and name.endswith("_moe"):
            final_score += 0.10  # Strong preference for MOE columns
        elif not want_moe and not name.endswith("_moe"):
            final_score += 0.05  # Preference for value columns
        elif want_moe and not name.endswith("_moe"):
            final_score -= 0.10  # Penalty for non-MOE when MOE is wanted
        elif not want_moe and name.endswith("_moe"):
            final_score -= 0.10  # Penalty for MOE when value is wanted
        
        # Check if this column matches any KEY_MAP base (bonus)
        for _, base in KEY_MAP_PATTERNS:
            if name == base or name == base + "_moe":
                final_score += 0.05
                break
        
        # Boost common important columns
        if any(important in name.lower() for important in ["total", "median", "population", "households"]):
            final_score += 0.02
            
        if final_score > best_score:
            best_col = name
            best_score = final_score
    
    # Only return if score meets threshold (tunable)
    if best_score >= SELECTION_THRESHOLD:
        return best_col
    
    # Fallback: return the best available column even if below threshold
    return best_col

def flip_moe(col: str, want_moe: bool, cols: Set[str]) -> str:
    """Convert between MOE and value columns based on intent."""
    if want_moe and not col.endswith("_moe"):
        if col + "_moe" in cols: return col + "_moe"
    if not want_moe and col.endswith("_moe"):
        base = col[:-4]
        if base in cols: return base
    return col

def repair_sql(sql: str, schema: Dict[str, Any], question: str, ctx: Dict[str, Any] = None) -> str:
    m = SELECT_COL_RE.search(sql)
    tbl = (schema.get("table_code") or schema.get("table") or "").lower()
    cols_set = {c["name"] for c in schema.get("columns", [])}

    if not m:
        col = best_column_for_question(schema, question) or choose_column(schema, question)
        # Use concrete geography instead of placeholder
        if ctx:
            target_geo, _, _, _ = infer_geo_from_ctx(ctx, question)
            is_multi_geo = isinstance(target_geo, list)
            if is_multi_geo:
                target_geo = [normalize_geo_for_table(geo, schema)[0] for geo in target_geo]
                geo_list = "','".join(target_geo)
                geo_predicate = f"geography_id IN ('{geo_list}')"
                select_cols = f"geography_id, {col}"
            else:
                target_geo = normalize_geo_for_table(target_geo, schema)[0]
                geo_predicate = f"geography_id = '{target_geo}'"
                select_cols = col
        else:
            geo_predicate = "geography_id = '01000US'"
            select_cols = col
        return f"SELECT {select_cols} FROM {tbl} WHERE {geo_predicate} AND year = 2022;"

    col_expr = normalize(m.group(1))
    if col_expr == "*" or "," in col_expr:
        col = best_column_for_question(schema, question) or choose_column(schema, question)
    else:
        col = col_expr

    col = flip_moe(col, wants_moe(question), cols_set)
    if col not in cols_set:
        col = best_column_for_question(schema, question) or choose_column(schema, question)

    fixed = re.sub(SELECT_COL_RE, f"SELECT {col} FROM {tbl} WHERE", sql)
    return fixed if fixed.endswith(";") else fixed + ";"

# ----------------------------- GEO enforcement -----------------------------
GEO_LENGTHS: Dict[str, int] = {"state": 2, "county": 5, "tract": 11, "block_group": 12}

# State name to FIPS mapping (lowercase full names to state summary IDs)
_STATE_NAME_TO_FIPS = {
    "alabama": "04000US01", "alaska": "04000US02", "arizona": "04000US04", "arkansas": "04000US05",
    "california": "04000US06", "colorado": "04000US08", "connecticut": "04000US09", "delaware": "04000US10",
    "florida": "04000US12", "georgia": "04000US13", "hawaii": "04000US15", "idaho": "04000US16",
    "illinois": "04000US17", "indiana": "04000US18", "iowa": "04000US19", "kansas": "04000US20",
    "kentucky": "04000US21", "louisiana": "04000US22", "maine": "04000US23", "maryland": "04000US24",
    "massachusetts": "04000US25", "michigan": "04000US26", "minnesota": "04000US27", "mississippi": "04000US28",
    "missouri": "04000US29", "montana": "04000US30", "nebraska": "04000US31", "nevada": "04000US32",
    "new hampshire": "04000US33", "new jersey": "04000US34", "new mexico": "04000US35", "new york": "04000US36",
    "north carolina": "04000US37", "north dakota": "04000US38", "ohio": "04000US39", "oklahoma": "04000US40",
    "oregon": "04000US41", "pennsylvania": "04000US42", "rhode island": "04000US44", "south carolina": "04000US45",
    "south dakota": "04000US46", "tennessee": "04000US47", "texas": "04000US48", "utah": "04000US49",
    "vermont": "04000US50", "virginia": "04000US51", "washington": "04000US53", "west virginia": "04000US54",
    "wisconsin": "04000US55", "wyoming": "04000US56", "district of columbia": "04000US11"
}

# USPS abbreviation to FIPS mapping (uppercase abbreviations to state summary IDs)
_USPS_ABBR_TO_FIPS = {
    "AL": "04000US01", "AK": "04000US02", "AZ": "04000US04", "AR": "04000US05", "CA": "04000US06",
    "CO": "04000US08", "CT": "04000US09", "DE": "04000US10", "FL": "04000US12", "GA": "04000US13",
    "HI": "04000US15", "ID": "04000US16", "IL": "04000US17", "IN": "04000US18", "IA": "04000US19",
    "KS": "04000US20", "KY": "04000US21", "LA": "04000US22", "ME": "04000US23", "MD": "04000US24",
    "MA": "04000US25", "MI": "04000US26", "MN": "04000US27", "MS": "04000US28", "MO": "04000US29",
    "MT": "04000US30", "NE": "04000US31", "NV": "04000US32", "NH": "04000US33", "NJ": "04000US34",
    "NM": "04000US35", "NY": "04000US36", "NC": "04000US37", "ND": "04000US38", "OH": "04000US39",
    "OK": "04000US40", "OR": "04000US41", "PA": "04000US42", "RI": "04000US44", "SC": "04000US45",
    "SD": "04000US46", "TN": "04000US47", "TX": "04000US48", "UT": "04000US49", "VT": "04000US50",
    "VA": "04000US51", "WA": "04000US53", "WV": "04000US54", "WI": "04000US55", "WY": "04000US56",
    "DC": "04000US11"
}

def infer_geo_from_ctx(ctx: dict, question: str = "") -> Tuple[Union[str, List[str]], str, List[str], str]:
    """
    Infer geography_id from context with comprehensive natural language parsing and conflict resolution.
    
    Returns:
        Tuple of (final_geography_id, resolution_method, resolved_geos, conflict_note)
        - final_geography_id: Single ID string or list of IDs for multi-state
        - resolution_method: Description of how the ID was resolved for logging
        - resolved_geos: List of all resolved geography IDs for telemetry
        - conflict_note: Description of any conflicts resolved
    """
    # 1. HIGHEST PRECEDENCE: Caller-provided geography_id
    if ctx.get("geography_id"):
        geo_id = str(ctx["geography_id"]).strip()
        if geo_id and geo_id != "None" and geo_id != "{GEO}":
            # Check for conflicts with question
            nl_result = _parse_geography_from_question(question)
            conflict_note = ""
            if nl_result:
                if isinstance(nl_result, list):
                    conflict_note = f"prompt_overridden_by_context: question specified {len(nl_result)} states"
                else:
                    conflict_note = "prompt_overridden_by_context: question specified single state"
            
            # Convert numeric FIPS to summary-style if needed
            if geo_id.isdigit():
                summary_id = _convert_numeric_to_summary(geo_id)
                if summary_id:
                    return summary_id, "explicit", [summary_id], conflict_note
                else:
                    return "01000US", "explicit-invalid-fallback", ["01000US"], conflict_note
            # If already summary-style, use as-is
            elif _is_summary_style_id(geo_id):
                return geo_id, "explicit", [geo_id], conflict_note
            else:
                return "01000US", "explicit-invalid-fallback", ["01000US"], conflict_note
    
    # 2. SECOND PRECEDENCE: FIPS components composition
    fips_result = _build_from_fips_components(ctx)
    if fips_result:
        return fips_result, "fips_compose", [fips_result], ""
    
    # 3. THIRD PRECEDENCE: Natural language parsing from question
    nl_result = _parse_geography_from_question(question)
    if nl_result:
        if isinstance(nl_result, list):
            # Multiple states found
            return nl_result, "multi_state", nl_result, ""
        else:
            # Single state found
            method = "nlu_fullname" if any(state_name in question.lower() for state_name in _STATE_NAME_TO_FIPS.keys()) else "nlu_usps"
            return nl_result, method, [nl_result], ""
    
    # 4. DEFAULT: National fallback
    return "01000US", "fallback_nation", ["01000US"], ""

def _convert_numeric_to_summary(numeric_fips: str) -> Optional[str]:
    """Convert numeric FIPS to appropriate summary-style ID."""
    if not numeric_fips.isdigit():
        return None
    
    length = len(numeric_fips)
    if length == 2:
        # State FIPS
        return f"04000US{numeric_fips.zfill(2)}"
    elif length == 5:
        # County FIPS (state + county)
        state_fips = numeric_fips[:2]
        county_fips = numeric_fips[2:]
        return f"05000US{state_fips.zfill(2)}{county_fips.zfill(3)}"
    elif length == 11:
        # Tract FIPS (state + county + tract)
        state_fips = numeric_fips[:2]
        county_fips = numeric_fips[2:5]
        tract_fips = numeric_fips[5:]
        return f"14000US{state_fips.zfill(2)}{county_fips.zfill(3)}{tract_fips.zfill(6)}"
    elif length == 12:
        # Block group FIPS (state + county + tract + block group)
        state_fips = numeric_fips[:2]
        county_fips = numeric_fips[2:5]
        tract_fips = numeric_fips[5:11]
        block_group = numeric_fips[11:]
        return f"15000US{state_fips.zfill(2)}{county_fips.zfill(3)}{tract_fips.zfill(6)}{block_group.zfill(1)}"
    
    return None

def _is_summary_style_id(geo_id: str) -> bool:
    """Check if geography_id is in summary-style format."""
    if not geo_id or len(geo_id) < 8:
        return False
    
    # Check for summary ID patterns: 01000US, 04000USss, 05000USssccc, etc.
    if geo_id.startswith("01000US") and len(geo_id) == 8:
        return True
    elif geo_id.startswith("04000US") and len(geo_id) == 10:
        return True
    elif geo_id.startswith("05000US") and len(geo_id) == 13:
        return True
    elif geo_id.startswith("14000US") and len(geo_id) == 19:
        return True
    elif geo_id.startswith("15000US") and len(geo_id) == 20:
        return True
    
    return False

def _build_from_fips_components(ctx: dict) -> Optional[str]:
    """Build summary-style geography_id from FIPS components."""
    state_fips = ctx.get("state_fips")
    county_fips = ctx.get("county_fips") 
    tract = ctx.get("tract")
    block_group = ctx.get("block_group")
    
    # State FIPS is required
    if not state_fips:
        return None
    
    try:
        st = str(state_fips).strip().zfill(2)
        if len(st) != 2 or not st.isdigit():
            return None
    except (ValueError, TypeError):
        return None
    
    # Build summary-style ID based on available components
    if block_group and tract and county_fips:
        # Block group level
        try:
            co = str(county_fips).strip().zfill(3)
            tr = str(tract).strip().zfill(6)
            bg = str(block_group).strip().zfill(1)
            if len(co) == 3 and co.isdigit() and len(tr) == 6 and tr.isdigit() and len(bg) == 1 and bg.isdigit():
                return f"15000US{st}{co}{tr}{bg}"
        except (ValueError, TypeError):
            pass
    
    if tract and county_fips:
        # Tract level
        try:
            co = str(county_fips).strip().zfill(3)
            tr = str(tract).strip().zfill(6)
            if len(co) == 3 and co.isdigit() and len(tr) == 6 and tr.isdigit():
                return f"14000US{st}{co}{tr}"
        except (ValueError, TypeError):
            pass
    
    if county_fips:
        # County level
        try:
            co = str(county_fips).strip().zfill(3)
            if len(co) == 3 and co.isdigit():
                return f"05000US{st}{co}"
        except (ValueError, TypeError):
            pass
    
    # State level
    return f"04000US{st}"

def _parse_geography_from_question(question: str) -> Optional[Union[str, List[str]]]:
    """Parse geography from natural language in the question. Returns single ID or list of IDs for multi-state."""
    if not question:
            return None
    
    question_lower = question.lower()
    
    # Check for national indicators
    national_patterns = ["us", "u.s.", "united states", "national", "nationwide", "country", "across the us"]
    for pattern in national_patterns:
        if re.search(rf"\b{re.escape(pattern)}\b", question_lower):
            return "01000US"
    
    # Check for "by state" without enumeration - fallback to national
    # PM DECISION: "by state" without explicit enumeration defaults to national
    # This prevents unexpected breakdowns and maintains deterministic behavior
    # Users wanting state breakdowns should explicitly list states or use "all states"
    if re.search(r"\bby\s+state\b", question_lower) and not _has_explicit_state_enumeration(question):
        return "01000US"
    
    # Check for multiple states first
    found_states = []
    
    # Find all full state names
    for state_name, summary_id in _STATE_NAME_TO_FIPS.items():
        if re.search(rf"\b{re.escape(state_name)}\b", question_lower):
            found_states.append(summary_id)
    
    # Find all USPS abbreviations with STRICT context requirements
    usps_pattern = r"\b([A-Z]{2})\b"
    matches = re.findall(usps_pattern, question)
    for match in matches:
        if match in _USPS_ABBR_TO_FIPS:
            # STRICT: Must be ALL-CAPS + context token (in|for|,|by|across) + word boundaries
            context_before = re.search(rf"\b(?:in|for|by|across|,)\s+{re.escape(match)}\b", question, re.I)
            context_after = re.search(rf"\b{re.escape(match)}\s+(?:state|,|\.|;|$)", question, re.I)
            if context_before or context_after:
                found_states.append(_USPS_ABBR_TO_FIPS[match])
    
    # Remove duplicates while preserving order (first occurrence wins)
    unique_states = []
    seen = set()
    for state in found_states:
        if state not in seen:
            unique_states.append(state)
            seen.add(state)
    
    # Cap at 15 states to prevent abuse
    # PM DECISION: 15-state cap based on UX considerations for charts/tables
    # For wider comparisons, users should use filters or request CSV export
    if len(unique_states) > 15:
        raise ValueError(
            f"Too many states specified ({len(unique_states)}). Maximum 15 states allowed. "
            f"For wider comparisons, please use filters or request a CSV export."
        )
    
    # Return results
    if len(unique_states) == 0:
        return None
    elif len(unique_states) == 1:
        return unique_states[0]
            else:
                pass
        return unique_states

def _has_explicit_state_enumeration(question: str) -> bool:
    """Check if question has explicit state enumeration (not just 'by state')."""
    question_lower = question.lower()
    
    # Check for explicit state names
    for state_name in _STATE_NAME_TO_FIPS.keys():
        if re.search(rf"\b{re.escape(state_name)}\b", question_lower):
            return True
    
    # Check for explicit USPS codes with context
    usps_pattern = r"\b([A-Z]{2})\b"
    matches = re.findall(usps_pattern, question)
    for match in matches:
        if match in _USPS_ABBR_TO_FIPS:
            context_before = re.search(rf"\b(?:in|for|by|across|,)\s+{re.escape(match)}\b", question, re.I)
            context_after = re.search(rf"\b{re.escape(match)}\s+(?:state|,|\.|;|$)", question, re.I)
            if context_before or context_after:
                return True
    
    return False

def _extract_year_from_question(question: str) -> Optional[int]:
    """Extract 4-digit year from the question."""
    if not question:
            return None
    
    # Look for 4-digit years (1900-2099)
    year_pattern = r"\b(19|20)\d{2}\b"
    matches = re.findall(year_pattern, question)
    if matches:
        # Return the last (most recent) year found
        return int(matches[-1] + matches[-1][-2:])  # Reconstruct full year
    return None

def _get_latest_year_from_schema(schema: Dict[str, Any]) -> int:
    """
    Get the latest year from the specific table schema deterministically.
    
    PERFORMANCE NOTE: This function only examines the provided schema object,
    never scans the entire schema directory or crosses table boundaries.
    """
    # Check for year in example_row or similar schema fields
    example_row = schema.get("example_row", {})
    if example_row and "year" in example_row:
        try:
            return int(example_row["year"])
        except (ValueError, TypeError):
            pass
    
    # Check for year in schema metadata
    schema_year = schema.get("year")
    if schema_year:
        try:
            return int(schema_year)
        except (ValueError, TypeError):
            pass
    
    # Check for year in columns (if any column has year in name/description)
    for col in schema.get("columns", []):
        col_name = col.get("name", "").lower()
        col_desc = col.get("description", "").lower()
        if "year" in col_name or "year" in col_desc:
            # Try to extract year from column name/description
            year_match = re.search(r"\b(19|20)\d{2}\b", col_name + " " + col_desc)
            if year_match:
                try:
                    return int(year_match.group(0))
                except (ValueError, TypeError):
                    pass
    
    # Default fallback
    return 2022

def pick_level_for_geo(geo_id: str) -> Optional[str]:
    """Determine geography level from summary-style or numeric geography_id."""
    if not geo_id:
        return None
    
    # Handle summary-style IDs
    if _is_summary_style_id(geo_id):
        if geo_id.startswith("01000US"):
            return "nation"
        elif geo_id.startswith("04000US"):
            return "state"
        elif geo_id.startswith("05000US"):
            return "county"
        elif geo_id.startswith("14000US"):
            return "tract"
        elif geo_id.startswith("15000US"):
            return "block_group"
    
    # Handle numeric FIPS (legacy support)
    if geo_id.isdigit():
        L = len(geo_id)
    for level, need in GEO_LENGTHS.items():
        if L == need:
            return level
    
    return None

def allowed_levels(schema: Dict[str, Any]) -> Set[str]:
    """Get allowed geography levels for a schema."""
    return set(schema.get("geography_levels", []))

def normalize_geo_for_table(geo_id: str, schema: Dict[str, Any]) -> Tuple[str, bool]:
    """Normalize geography_id for a specific table with deterministic schema level compliance."""
    if not geo_id or geo_id == "{GEO}":
        return "01000US", False  # Default to national fallback
    
    # Clean and validate input
    geo_id = str(geo_id).strip()
    
    # Convert numeric FIPS to summary-style if needed
    if geo_id.isdigit():
        geo_id = _convert_numeric_to_summary(geo_id) or "01000US"
    
    # Determine geography level
    lvl = pick_level_for_geo(geo_id)
    if not lvl:
        print(f"Warning: Unrecognized geography_id format: '{geo_id}'. Using national fallback.")
        return "01000US", False
    
    # Check if level is allowed for this table
    allowed = allowed_levels(schema)
    if not allowed:
        # No restrictions - use as-is
        return geo_id, False
    
    if lvl in allowed:
        # Level is allowed - use as-is
        return geo_id, False
    
    # Level not allowed - apply deterministic ladder
    adjusted_geo = _apply_schema_level_ladder(geo_id, lvl, allowed)
    if adjusted_geo != geo_id:
        table_name = schema.get('table_code') or schema.get('table') or 'unknown'
        print(f"Warning: Geo level '{lvl}' not in allowed {sorted(allowed)} for table {table_name}. Adjusted to {adjusted_geo}.")
        return adjusted_geo, True
    
    return geo_id, False

def _apply_schema_level_ladder(geo_id: str, current_level: str, allowed_levels: Set[str]) -> str:
    """
    Apply deterministic schema level compliance ladder.
    
    SCHEMA LADDER SEMANTICS (PM DOCUMENTED):
    ======================================
    
    Ladder Order (most general → most specific):
    nation → state → county → tract → block_group
    
    Step-up Strategy (preferred):
    1. Try more general levels first (nation, state, county, tract)
    2. Only if no general level allowed, try more specific levels
    
    Examples per table class:
    - Nation-only tables: nation disallowed → fallback to 01000US (error case)
    - State+County tables: tract requested → step up to county (if allowed) or state
    - State-only tables: county requested → step up to state
    - County-only tables: state requested → step down to county (if allowed)
    
    Deterministic mapping ensures consistent behavior across all table types.
    """
    # Define the ladder order (most general to most specific)
    ladder = ["nation", "state", "county", "tract", "block_group"]
    
    # Find current level position
    try:
        current_pos = ladder.index(current_level)
    except ValueError:
        # Unknown level - fallback to nation
        return "01000US"
    
    # Try to step up to more general levels first
    for i in range(current_pos - 1, -1, -1):
        if ladder[i] in allowed_levels:
            return _convert_to_level(geo_id, ladder[i])
    
    # If no more general level allowed, try more specific levels
    for i in range(current_pos + 1, len(ladder)):
        if ladder[i] in allowed_levels:
            return _convert_to_level(geo_id, ladder[i])
    
    # If nothing works, fallback to nation
    return "01000US"

def _convert_to_level(geo_id: str, target_level: str) -> str:
    """Convert geography_id to a specific level."""
    if target_level == "nation":
        return "01000US"
    elif target_level == "state":
        # Extract state FIPS from any level
        if geo_id.startswith("04000US"):
    return geo_id
        elif geo_id.startswith("05000US"):
            return geo_id[:10]  # Keep state part
        elif geo_id.startswith("14000US"):
            return geo_id[:10]  # Keep state part
        elif geo_id.startswith("15000US"):
            return geo_id[:10]  # Keep state part
        else:
            return "01000US"
    elif target_level == "county":
        # Extract county FIPS from any level
        if geo_id.startswith("05000US"):
            return geo_id
        elif geo_id.startswith("14000US"):
            return geo_id[:13]  # Keep county part
        elif geo_id.startswith("15000US"):
            return geo_id[:13]  # Keep county part
        else:
            return "01000US"
    elif target_level == "tract":
        # Extract tract FIPS from any level
        if geo_id.startswith("14000US"):
            return geo_id
        elif geo_id.startswith("15000US"):
            return geo_id[:19]  # Keep tract part
        else:
            return "01000US"
    elif target_level == "block_group":
        # Only block group level can be used as-is
        if geo_id.startswith("15000US"):
            return geo_id
        else:
            return "01000US"
    
    return "01000US"

def _validate_geography_ids(geo_ids: List[str]) -> List[str]:
    """Validate geography IDs against strict allow-list."""
    valid_ids = []
    invalid_ids = []
    
    for geo_id in geo_ids:
        if _is_summary_style_id(geo_id):
            valid_ids.append(geo_id)
        else:
            invalid_ids.append(geo_id)
            print(f"Warning: Invalid geography ID '{geo_id}' filtered out")
    
    if not valid_ids:
        # User-actionable error message
        invalid_list = ", ".join(invalid_ids[:5])  # Show first 5 invalid IDs
        if len(invalid_ids) > 5:
            invalid_list += f" (and {len(invalid_ids) - 5} more)"
        
        raise ValueError(
            f"No valid geography IDs found. Invalid IDs: {invalid_list}. "
            f"Please use valid state names (e.g., 'California', 'Texas') or USPS codes (e.g., 'CA', 'TX')."
        )
    
    return valid_ids

_GEO_CLAUSE_RE        = re.compile(r"(?is)\bwhere\b(.+)$")
_GEO_ID_REPLACER      = re.compile(r"(?is)\bgeography_id\s*=\s*('[^']*'|\{GEO\})")
_GEO_ID_IN_REPLACER   = re.compile(r"(?is)\bgeography_id\s+in\s*\((.*?)\)")
_CONFLICTING_PREDICATES = [
    r"\bstate_fips\s*=\s*'?\d{1,2}'?",
    r"\bcounty_fips\s*=\s*'?\d{1,3}'?",
    r"\bstate\s*=\s*'?[A-Z]{2}'?",
    r"\bstusab\s*=\s*'?[A-Z]{2}'?",
]

def enforce_geo(sql: str, schema: Dict[str, Any], ctx: Dict[str, Any], question: str = "", geo_column: str = "geography_id") -> Tuple[str, Dict[str, Any]]:
    """Enforce geography constraints in SQL with comprehensive telemetry and deterministic behavior."""
    try:
        # Determine target geography using comprehensive inference
        target_geo, resolution_method, resolved_geos, conflict_note = infer_geo_from_ctx(ctx, question)
        
        # Handle multi-geography case with strict validation
        is_multi_geo = isinstance(target_geo, list)
        schema_level_adjusted = False
        
        if is_multi_geo:
            # Validate all geographies against strict allow-list
            try:
                validated_geos = _validate_geography_ids(target_geo)
            except ValueError as e:
                print(f"Warning: {e}. Using national fallback.")
                validated_geos = ["01000US"]
                is_multi_geo = False
                resolution_method = "fallback_nation"
            
            # Normalize all geographies for the specific table
            normalized_geos = []
            for geo in validated_geos:
                try:
                    normalized_geo, adjusted = normalize_geo_for_table(geo, schema)
                    normalized_geos.append(normalized_geo)
                    if adjusted:
                        schema_level_adjusted = True
                except Exception as e:
                    print(f"Warning: Failed to normalize geography {geo}: {e}")
                    normalized_geos.append("01000US")
            
            target_geo = normalized_geos
        else:
            # Single geography - normalize for the specific table
            try:
                target_geo, adjusted = normalize_geo_for_table(target_geo, schema)
                if adjusted:
                    schema_level_adjusted = True
            except Exception as e:
                print(f"Warning: Failed to normalize geography {target_geo}: {e}")
                target_geo = "01000US"
                resolution_method = "fallback_nation"
        
        # Determine year with deterministic priority
        year = _extract_year_from_question(question)
        year_source = "prompt" if year else "context" if ctx.get("year") else "schema"
        if not year:
            year = ctx.get("year") or _get_latest_year_from_schema(schema)
        
        # Log the resolution method for telemetry
        print(f"Geography resolved via: {resolution_method} -> {target_geo}")
        
        # Surface conflicts in logs for monitoring
        if conflict_note:
            print(f"CONFLICT RESOLVED: {conflict_note}")
            print(f"Caller context geography_id took precedence over question text")
        
        # Build geography predicate
        if is_multi_geo:
            geo_list = "','".join(target_geo)
            geo_predicate = f"{geo_column} IN ('{geo_list}')"
            # PERFORMANCE NOTE: IN with 15 items should be fine for most DBs
            # If performance issues arise, consider VALUES (...) JOIN strategy
        else:
            geo_predicate = f"{geo_column} = '{target_geo}'"
        
        # Find existing WHERE clause
        m = _GEO_CLAUSE_RE.search(sql)
        if not m:
            # No WHERE clause - add one with geography and year
            where_clause = f"{geo_predicate} AND year = {year}"
            final_sql = sql.rstrip(" ;") + f" WHERE {where_clause};"
        else:
        where_rest = m.group(1)

        # Replace existing geography_id clauses
        def _replace_eq(_):
                return geo_predicate
        where_rest = _GEO_ID_REPLACER.sub(_replace_eq, where_rest)

        # Replace geography_id IN clauses
        if _GEO_ID_IN_REPLACER.search(where_rest):
                where_rest = _GEO_ID_IN_REPLACER.sub(geo_predicate, where_rest)

        # Add geography_id if missing
            if not re.search(rf"(?is)\b{re.escape(geo_column)}\s*(?:=|IN)", where_rest):
                where_rest = f"{geo_predicate} AND {where_rest.strip()}"

            # Remove conflicting geography predicates (state_fips, county_fips, etc.)
        for pat in _CONFLICTING_PREDICATES:
            where_rest = re.sub(rf"(?is)\bAND\s*({pat})\s*(?:AND|$)", " AND ", where_rest)
            where_rest = re.sub(rf"(?is)^\s*({pat})\s*(?:AND|$)", "", where_rest)

            # Ensure year constraint is present and correct
            year_pattern = r"\byear\s*=\s*\d+"
            if re.search(year_pattern, where_rest, re.I):
                # Replace existing year constraint
                where_rest = re.sub(year_pattern, f"year = {year}", where_rest, flags=re.I)
            else:
                # Add year constraint
                where_rest = f"{where_rest.strip()} AND year = {year}"

        # Reconstruct SQL
            final_sql = _GEO_CLAUSE_RE.sub("WHERE " + where_rest.strip(), sql)
            final_sql = re.sub(r"(?is)\s+AND\s+AND\s+", " AND ", final_sql)
            final_sql = re.sub(r"\s{2,}", " ", final_sql).strip()
        
        # Ensure semicolon
            if not final_sql.endswith(";"):
                final_sql += ";"
        
        # Build comprehensive telemetry data
        telemetry = {
            "resolution_method": resolution_method,
            "resolved_geos": resolved_geos,
            "moe": wants_moe(question) if question else False,
            "is_multi_geo": is_multi_geo,
            "year": year,
            "year_source": year_source,
            "schema_level_used": pick_level_for_geo(target_geo[0] if is_multi_geo else target_geo),
            "schema_level_adjusted": schema_level_adjusted,
            "conflict_note": conflict_note
        }
        
        return final_sql, telemetry
        
    except Exception as e:
        # Fallback: return SQL with national fallback
        print(f"Warning: Geography enforcement failed: {e}. Using national fallback.")
        year = _extract_year_from_question(question) or ctx.get("year") or _get_latest_year_from_schema(schema)
        where_clause = f"{geo_column} = '01000US' AND year = {year}"
        
        # Try to add WHERE clause or replace existing one
        if not _GEO_CLAUSE_RE.search(sql):
            final_sql = sql.rstrip(" ;") + f" WHERE {where_clause};"
        else:
            # Replace existing WHERE clause
            final_sql = _GEO_CLAUSE_RE.sub(f"WHERE {where_clause}", sql)
        
        # Fallback telemetry
        telemetry = {
            "resolution_method": "fallback_nation",
            "resolved_geos": ["01000US"],
            "moe": wants_moe(question) if question else False,
            "is_multi_geo": False,
            "year": year,
            "year_source": "fallback",
            "schema_level_used": "nation",
            "schema_level_adjusted": False,
            "conflict_note": f"error_fallback: {str(e)}"
        }
        
        return final_sql, telemetry

def _ensure_geography_id_in_select(sql: str, is_multi_geo: bool) -> str:
    """
    Ensure geography_id is included in SELECT for all queries.
    
    PM DECISION: Always include geography_id in SELECT to simplify downstream joins/visualizations
    and avoid conditional schemas. This provides consistent output format regardless of
    single vs multi-geography queries.
    """
    # Always include geography_id in SELECT (PM decision for consistent schemas)
    
    # Extract SELECT clause
    select_match = re.search(r"(?is)^\s*select\s+(.+?)\s+from\s+", sql)
    if not select_match:
        return sql
    
    select_clause = select_match.group(1).strip()
    
    # Check if geography_id is already in SELECT
    if re.search(r"(?is)\bgeography_id\b", select_clause):
        return sql
    
    # Add geography_id to SELECT clause
    if select_clause == "*":
        # Replace * with geography_id, *
        new_select = "geography_id, *"
    else:
        # Add geography_id at the beginning
        new_select = f"geography_id, {select_clause}"
    
    # Replace the SELECT clause
    new_sql = re.sub(r"(?is)^\s*select\s+.+?\s+from\s+", f"SELECT {new_select} FROM ", sql)
    return new_sql

# ----------------------------- Model init -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

t5_tok = AutoTokenizer.from_pretrained(str(T5_DIR))
t5_mdl = AutoModelForSeq2SeqLM.from_pretrained(str(T5_DIR)).to(device)

sql_tok = AutoTokenizer.from_pretrained(str(SQL_DIR), use_fast=True)
if sql_tok.pad_token is None:
    sql_tok.pad_token = sql_tok.eos_token
sql_mdl = AutoModelForCausalLM.from_pretrained(
    str(SQL_DIR),
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32
).to(device)
sql_mdl.config.pad_token_id = sql_tok.pad_token_id

# discourage JSON/DDL/markdown tokens for decoder-only model
bad_strs = ["{", "}", "[", "]", "CREATE TABLE", "columns", "table_code", "sql_ddl", ":'", "```", "SELECT *"]
bad_ids = [sql_tok(b, add_special_tokens=False).input_ids for b in bad_strs]
logits_processors = LogitsProcessorList([NoBadWordsLogitsProcessor(bad_ids, eos_token_id=sql_tok.eos_token_id)])

# ----------------------------- Main loop -----------------------------
def process_test_case(test_case: Dict[str, Any], t5_tok: Any, t5_mdl: Any, sql_tok: Any, sql_mdl: Any, logits_processors: Any) -> Dict[str, str]:
    """Process a single test case with comprehensive error handling."""
    try:
        # Load schema with error handling
        try:
            schema = load_schema(test_case["table"])
        except (FileNotFoundError, json.JSONDecodeError) as e:
            print(f"Warning: Failed to load schema for {test_case['table']}: {e}")
            # Create minimal fallback schema
            schema = {
                "table_code": test_case["table"],
                "columns": [{"name": "total_population", "description": "Total population"}],
                "geography_levels": ["state", "county"]
            }
        
        tbl = (schema.get("table_code") or schema.get("table") or "").lower()

        # Build prompt
        prompt = build_schema_aware_prompt(schema, test_case["question"])
        prompt += "\nRules:\n- Use only the geography_id and year specified in the question; ignore any example_row values.\n"
        prompt += FORMAT_CONTRACT

        # Build context for geography inference
        ctx = {
            "geography_id": test_case.get("geography_id"),
            "state_fips":   test_case.get("state_fips"),
            "county_fips":  test_case.get("county_fips"),
            "tract":        test_case.get("tract"),
            "block_group":  test_case.get("block_group"),
            "year":         test_case.get("year", 2022),
        }

        # Generate T5 SQL with error handling
        try:
            enc = t5_tok(prompt, return_tensors="pt", truncation=True, padding=True, max_length=512).to(t5_mdl.device)
            out = t5_mdl.generate(
                **enc,
                max_new_tokens=160,
                do_sample=False,
                num_beams=1,
                early_stopping=True,
                eos_token_id=t5_tok.eos_token_id
            )
            t5_full = t5_tok.decode(out[0], skip_special_tokens=True)
            sql_t5 = extract_sql(t5_full, prompt)
        except Exception as e:
            print(f"Warning: T5 generation failed for {test_case['table']}: {e}")
            sql_t5 = build_output_sql(schema, test_case["question"], ctx)

        # Generate SQLCoder SQL with error handling
        try:
            enc2 = sql_tok(prompt, return_tensors="pt", truncation=True, padding=True, max_length=2048).to(sql_mdl.device)
            out2 = sql_mdl.generate(
                **enc2,
                max_new_tokens=160,
                do_sample=False,
                num_beams=1,
                temperature=0.0,
                repetition_penalty=1.05,
                logits_processor=logits_processors
            )
            sql_full = sql_tok.decode(out2[0], skip_special_tokens=True)
            sql_sqlcoder = extract_sql(sql_full, prompt)
        except Exception as e:
            print(f"Warning: SQLCoder generation failed for {test_case['table']}: {e}")
            sql_sqlcoder = build_output_sql(schema, test_case["question"], ctx)

        # Repair SQL with error handling
        try:
            sql_t5 = repair_sql(sql_t5, schema, test_case["question"], ctx)
            sql_sqlcoder = repair_sql(sql_sqlcoder, schema, test_case["question"], ctx)
        except Exception as e:
            print(f"Warning: SQL repair failed for {test_case['table']}: {e}")

        # Geography enforcement with question context
        t5_telemetry = {}
        sqlcoder_telemetry = {}
        try:
            sql_t5, t5_telemetry = enforce_geo(sql_t5, schema, ctx, test_case["question"])
            sql_sqlcoder, sqlcoder_telemetry = enforce_geo(sql_sqlcoder, schema, ctx, test_case["question"])
            
            # Ensure geography_id is in SELECT for all queries (PM decision for consistent schemas)
            sql_t5 = _ensure_geography_id_in_select(sql_t5, t5_telemetry.get("is_multi_geo", False))
            sql_sqlcoder = _ensure_geography_id_in_select(sql_sqlcoder, sqlcoder_telemetry.get("is_multi_geo", False))
        except Exception as e:
            print(f"Warning: Geography enforcement failed for {test_case['table']}: {e}")

        # Final validation with fallback
        if not is_valid_sql(sql_t5, tbl):
            try:
                sql_t5 = build_output_sql(schema, test_case["question"], ctx)
                sql_t5, t5_telemetry = enforce_geo(sql_t5, schema, ctx, test_case["question"])
                sql_t5 = _ensure_geography_id_in_select(sql_t5, t5_telemetry.get("is_multi_geo", False))
            except Exception as e:
                print(f"Warning: T5 fallback failed for {test_case['table']}: {e}")
                sql_t5 = f"SELECT total_population FROM {tbl} WHERE geography_id = '01000US' AND year = 2022;"
                t5_telemetry = {"resolution_method": "fallback_nation", "resolved_geos": ["01000US"], "moe": False, "is_multi_geo": False, "year": 2022}

        if not is_valid_sql(sql_sqlcoder, tbl):
            try:
                sql_sqlcoder = build_output_sql(schema, test_case["question"], ctx)
                sql_sqlcoder, sqlcoder_telemetry = enforce_geo(sql_sqlcoder, schema, ctx, test_case["question"])
                sql_sqlcoder = _ensure_geography_id_in_select(sql_sqlcoder, sqlcoder_telemetry.get("is_multi_geo", False))
            except Exception as e:
                print(f"Warning: SQLCoder fallback failed for {test_case['table']}: {e}")
                sql_sqlcoder = f"SELECT total_population FROM {tbl} WHERE geography_id = '01000US' AND year = 2022;"
                sqlcoder_telemetry = {"resolution_method": "fallback_nation", "resolved_geos": ["01000US"], "moe": False, "is_multi_geo": False, "year": 2022}

        return {
            "table": test_case["table"],
            "question": test_case["question"],
            "t5_sql": sql_t5,
            "sqlcoder_sql": sql_sqlcoder,
            "t5_telemetry": t5_telemetry,
            "sqlcoder_telemetry": sqlcoder_telemetry
        }
        
    except Exception as e:
        print(f"Error processing test case {test_case.get('table', 'unknown')}: {e}")
        # Return minimal fallback
        return {
            "table": test_case.get("table", "unknown"),
            "question": test_case.get("question", ""),
            "t5_sql": "SELECT total_population FROM unknown WHERE geography_id = '01000US' AND year = 2022;",
            "sqlcoder_sql": "SELECT total_population FROM unknown WHERE geography_id = '01000US' AND year = 2022;",
            "t5_telemetry": {"resolution_method": "fallback_nation", "resolved_geos": ["01000US"], "moe": False, "is_multi_geo": False, "year": 2022},
            "sqlcoder_telemetry": {"resolution_method": "fallback_nation", "resolved_geos": ["01000US"], "moe": False, "is_multi_geo": False, "year": 2022}
        }

# Main execution
cnt = 0
OUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    with open(OUT_PATH, "w") as sink:
        for t in tests:
            result = process_test_case(t, t5_tok, t5_mdl, sql_tok, sql_mdl, logits_processors)
            sink.write(json.dumps(result) + "\n")
            cnt += 1
            
            # Progress indicator
            if cnt % 10 == 0:
                print(f"Processed {cnt} test cases...")
                
except Exception as e:
    print(f"Fatal error in main loop: {e}")
    raise

# ----------------------------- Cleanup -----------------------------
del t5_mdl, sql_mdl; gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f"Wrote {cnt} rows → {OUT_PATH}")

# ----------------------------- Mini Synonym Test Suite -----------------------------
"""
SYNONYM TEST SUITE - Run with: python ab_infer_hardened.py --test-synonyms

This test suite validates enhanced synonym recognition across hot tables.
It tests alias expansion, MOE detection, and hybrid column selection without
hitting external I/O or changing main execution behavior.

To run: Set environment variable RUN_SYNONYM_TESTS=1 or use --test-synonyms flag
To tune: Adjust SELECTION_THRESHOLD and TIE_DELTA constants at top of file
"""

def create_mock_schema(table_code: str) -> Dict[str, Any]:
    """Create mock schema for testing without external I/O."""
    schemas = {
        "B01001": {
            "table_code": "b01001",
            "columns": [
                {"name": "total_population", "description": "Total population"},
                {"name": "total_population_moe", "description": "Margin of error for total population"},
                {"name": "male_total", "description": "Male total population"},
                {"name": "male_total_moe", "description": "Margin of error for male total"},
                {"name": "female_population", "description": "Female population"},
                {"name": "female_population_moe", "description": "Margin of error for female population"},
            ]
        },
        "B02001": {
            "table_code": "b02001", 
            "columns": [
                {"name": "total_population", "description": "Total population"},
                {"name": "total_population_moe", "description": "Margin of error for total population"},
                {"name": "white_alone", "description": "White alone population"},
                {"name": "white_alone_moe", "description": "Margin of error for white alone"},
                {"name": "black_or_african_american_alone", "description": "Black or African American alone"},
                {"name": "black_or_african_american_alone_moe", "description": "Margin of error for black or African American alone"},
                {"name": "asian_alone", "description": "Asian alone population"},
                {"name": "asian_alone_moe", "description": "Margin of error for Asian alone"},
            ]
        },
        "B19001": {
            "table_code": "b19001",
            "columns": [
                {"name": "total_households", "description": "Total number of households"},
                {"name": "total_households_moe", "description": "Margin of error for total households"},
                {"name": "median_household_income", "description": "Median household income"},
                {"name": "median_household_income_moe", "description": "Margin of error for median household income"},
            ]
        },
        "B23025": {
            "table_code": "b23025",
            "columns": [
                {"name": "total_population_16_over", "description": "Total population 16 years and over"},
                {"name": "total_population_16_over_moe", "description": "Margin of error for total population 16 years and over"},
                {"name": "civilian_labor_force", "description": "Civilian labor force"},
                {"name": "civilian_labor_force_moe", "description": "Margin of error for civilian labor force"},
                {"name": "unemployed", "description": "Unemployed population"},
                {"name": "unemployed_moe", "description": "Margin of error for unemployed population"},
            ]
        },
        "B24010": {
            "table_code": "b24010",
            "columns": [
                {"name": "male_service", "description": "Male service occupations"},
                {"name": "male_service_moe", "description": "Margin of error for male service occupations"},
                {"name": "female_service", "description": "Female service occupations"},
                {"name": "female_service_moe", "description": "Margin of error for female service occupations"},
            ]
        },
        "B15003": {
            "table_code": "b15003",
            "columns": [
                {"name": "total_population_25_over", "description": "Total population 25 years and over"},
                {"name": "total_population_25_over_moe", "description": "Margin of error for total population 25 years and over"},
                {"name": "high_school_graduate_or_higher", "description": "High school graduate or higher"},
                {"name": "high_school_graduate_or_higher_moe", "description": "Margin of error for high school graduate or higher"},
                {"name": "bachelors_degree", "description": "Bachelor's degree"},
                {"name": "bachelors_degree_moe", "description": "Margin of error for bachelor's degree"},
            ]
        },
        "B14001": {
            "table_code": "b14001",
            "columns": [
                {"name": "total_population_3plus", "description": "Total population 3 years and over"},
                {"name": "total_population_3plus_moe", "description": "Margin of error for total population 3 years and over"},
                {"name": "nursery_preschool", "description": "Nursery school, preschool"},
                {"name": "nursery_preschool_moe", "description": "Margin of error for nursery school, preschool"},
                {"name": "male_not_enrolled", "description": "Male not enrolled in school"},
                {"name": "male_not_enrolled_moe", "description": "Margin of error for male not enrolled"},
            ]
        },
        "DP02": {
            "table_code": "dp02",
            "columns": [
                {"name": "high_school_graduate_or_higher", "description": "High school graduate or higher"},
                {"name": "high_school_graduate_or_higher_moe", "description": "Margin of error for high school graduate or higher"},
                {"name": "bachelors_degree", "description": "Bachelor's degree"},
                {"name": "bachelors_degree_moe", "description": "Margin of error for bachelor's degree"},
            ]
        }
    }
    return schemas.get(table_code, {"table_code": table_code.lower(), "columns": []})

def run_synonym_tests() -> bool:
    """Run comprehensive synonym tests and return success status."""
    print("🧪 Running Synonym Test Suite...")
    print(f"   RapidFuzz available: {_HAS_RF}")
    print(f"   Selection threshold: {SELECTION_THRESHOLD}")
    print(f"   Tie delta: {TIE_DELTA}")
    print()
    
    test_cases = [
        # B01001 - Population demographics
        ("B01001", "What is the total population?", "total_population"),
        ("B01001", "What is the margin of error for total population?", "total_population_moe"),
        ("B01001", "How many women are there?", "female_population"),
        ("B01001", "What's the uncertainty in female population?", "female_population_moe"),
        ("B01001", "Show me male total", "male_total"),
        
        # B02001 - Race and ethnicity
        ("B02001", "How many people are black alone?", "black_or_african_american_alone"),
        ("B02001", "What's the error margin for black alone population?", "black_or_african_american_alone_moe"),
        ("B02001", "Show me white alone population", "white_alone"),
        ("B02001", "What's the uncertainty in Asian alone?", "asian_alone_moe"),
        
        # B19001 - Income and households
        ("B19001", "What is the total number of households?", "total_households"),
        ("B19001", "What's the error margin for total households?", "total_households_moe"),
        ("B19001", "Show me median household income", "median_household_income"),
        ("B19001", "What's the uncertainty in median income?", "median_household_income_moe"),
        
        # B23025 - Employment and age
        ("B23025", "How many people are 16+?", "total_population_16_over"),
        ("B23025", "What's the margin of error for population 16 and over?", "total_population_16_over_moe"),
        ("B23025", "Show me civilian labor force", "civilian_labor_force"),
        ("B23025", "What's the uncertainty in unemployed population?", "unemployed_moe"),
        
        # B24010 - Service occupations
        ("B24010", "How many men are in service occupations?", "male_service"),
        ("B24010", "What's the error margin for male service?", "male_service_moe"),
        ("B24010", "Show me female service occupations", "female_service"),
        
        # B15003 - Education
        ("B15003", "How many people have HS grad or higher?", "high_school_graduate_or_higher"),
        ("B15003", "What's the uncertainty in high school graduate or higher?", "high_school_graduate_or_higher_moe"),
        ("B15003", "Show me bachelor's degree holders", "bachelors_degree"),
        ("B15003", "What's the error margin for bachelor's degree?", "bachelors_degree_moe"),
        
        # B14001 - Early education
        ("B14001", "How many kids are in pre-k?", "nursery_preschool"),
        ("B14001", "What's the margin of error for nursery preschool?", "nursery_preschool_moe"),
        ("B14001", "Show me male not enrolled", "male_not_enrolled"),
        ("B14001", "What's the uncertainty in male not enrolled?", "male_not_enrolled_moe"),
        
        # DP02 - Education summary
        ("DP02", "How many people have HS+ education?", "high_school_graduate_or_higher"),
        ("DP02", "What's the error margin for HS graduate or higher?", "high_school_graduate_or_higher_moe"),
    ]
    
    passed = 0
    failed = 0
    
    for table_code, question, expected in test_cases:
        schema = create_mock_schema(table_code)
        result = best_column_for_question(schema, question)
        
        # Check if result matches expected or its MOE twin
        expected_moe = expected + "_moe" if not expected.endswith("_moe") else expected[:-4]
        is_match = (result == expected or result == expected_moe)
        
        status = "✅" if is_match else "❌"
        print(f"{status} {table_code}: '{question}'")
        print(f"    Expected: {expected} (or {expected_moe})")
        print(f"    Got:      {result}")
        
        if is_match:
            passed += 1
        else:
            failed += 1
        print()
    
    print(f"📊 Test Results: {passed} passed, {failed} failed")
    
    if failed == 0:
        print("🎉 All synonym tests passed!")
        return True
    else:
        print(f"⚠️  {failed} tests failed. Consider tuning SELECTION_THRESHOLD or TIE_DELTA.")
        return False

# ----------------------------- Main Runner -----------------------------

def run_ab_inference():
    cnt = 0
    OUT_DIR.mkdir(parents=True, exist_ok=True)

        # T5
    t5_tok, t5_mdl = AutoTokenizer.from_pretrained(str(T5_DIR)), \
                     AutoModelForSeq2SeqLM.from_pretrained(str(T5_DIR)).to(device)
    
    # SQLCoder
    sql_tok, sql_mdl = AutoTokenizer.from_pretrained(str(SQL_DIR)), \
                       AutoModelForCausalLM.from_pretrained(
                           str(SQL_DIR),
                           torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32
                       ).to(device)



    try:
        with open(OUT_PATH, "w") as sink:
            for t in tests:
                result = process_test_case(t, t5_tok, t5_mdl, sql_tok, sql_mdl, logits_processors)
                sink.write(json.dumps(result) + "\n")
                cnt += 1
                if cnt % 10 == 0:
                    print(f"Processed {cnt} test cases...")
    except Exception as e:
        print(f"Fatal error in main loop: {e}")
        raise

    # Cleanup
    del t5_mdl, sql_mdl; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"Wrote {cnt} rows → {OUT_PATH}")


# ----------------------------- Notebook Entry -----------------------------

# Always run inference when this script is run inside a notebook
run_ab_inference()

import os
if os.environ.get("RUN_SYNONYM_TESTS") == "1":
    run_synonym_tests()



Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Processed 10 test cases...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Processed 20 test cases...


...

Processed 20 test cases...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Processed 30 test cases...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Wrote 35 rows → /mnt/govquery-volume/models/outputs/ab_compare_all.jsonl
🧪 Running Synonym Test Suite...
   RapidFuzz available: True
   Selection threshold: 0.62
   Tie delta: 0.02

✅ B01001: 'What is the total population?'
    Expected: total_population (or total_population_moe)
    Got:      total_population

✅ B01001: 'What is the margin of error for total population?'
    Expected: total_population_moe (or total_population)
    Got:      total_population_moe

✅ B01001: 'How many women are there?'
    Expected: female_population (or female_population_moe)
    Got:      female_population

✅ B01001: 'What's the uncertainty in female population?'
    Expected: female_population_moe (or female_population)
    Got:      female_population_moe

✅ B01001: 'Show me male total'
    Expected: male_total (or male_total_moe)
    Got:      male_total

✅ B02001: 'How many people are black alone?'
    Expected: black_or_african_american_alone (or black_or_african_american_alone_moe)
    Got:      

In [13]:
import os
os.environ['RUN_SYNONYM_TESTS'] = '1'

In [20]:
import os
print("T5_DIR exists:", os.path.isdir("/mnt/govquery-volume/models/t5_lora"))
print("SQL_DIR exists:", os.path.isdir("/mnt/govquery-volume/models/sqlcoder_lora_fp"))


T5_DIR exists: True
SQL_DIR exists: True


# Syntax Validation

In [20]:
# --- Column-existence validation with normalization ---
import re, json
from pathlib import Path

SCHEMAS_DIR = Path("/mnt/govquery-volume/schemas")
OUT_DIR = Path("/mnt/govquery-volume/models/outputs")
RESULTS = OUT_DIR/"ab_compare_all.jsonl"          # or "ab_compare_all_guarded.jsonl"

def load_cols(code):
    with open(SCHEMAS_DIR/f"{code}.json") as f: s = json.load(f)
    # return bare column names from schema
    return {c["name"] for c in s["columns"]}

# normalize SELECT list into bare column names
def extract_cols(sql: str):
    m = re.search(r"SELECT\s+(.*?)\s+FROM", sql, flags=re.IGNORECASE|re.DOTALL)
    if not m: return set()
    raw = m.group(1)

    # split by commas at top level (very naive but OK here)
    parts = [p.strip() for p in raw.split(",") if p.strip()]
    cols = []
    for p in parts:
        # remove simple aggregates/casts
        p = re.sub(r"\b(CAST|SUM|COUNT|AVG|MIN|MAX)\s*\((.*?)\)", r"\2", p, flags=re.IGNORECASE)
        # remove aliases
        p = re.sub(r"\bas\b.*$", "", p, flags=re.IGNORECASE).strip()
        # strip quotes/backticks
        p = p.strip('`"')
        # drop wildcard
        if p == "*": 
            continue
        # strip table qualifier: b02001.col -> col (handles a.b or "a"."b")
        if "." in p:
            p = p.split(".")[-1].strip('`"')
        cols.append(p)
    return set(cols)

violations = []
with open(RESULTS) as f:
    for line in f:
        r = json.loads(line)
        valid = load_cols(r["table"])
        for tag in ("t5_sql","sqlcoder_sql"):
            used = extract_cols(r[tag])
            miss = used - valid
            if miss:
                violations.append({"table": r["table"], "model": tag, "question": r["question"], "missing": sorted(miss), "sql": r[tag]})

report_path = OUT_DIR/"validation_report_normalized.json"
with open(report_path, "w") as f:
    json.dump(violations, f, indent=2)

print(f"{'No violations 🎉' if not violations else f'Found ' + str(len(violations)) + ' issues → ' + str(report_path)}")


No violations 🎉


# Main Script (1. Semantic Training) (2. SQL Model Training 2)

In [18]:
# =========================
# Novice Mode: intent → slots → guarded SQL
# =========================
import re, json, gc, torch
from pathlib import Path
from typing import Dict, Any, Tuple, Optional
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM

# --- Paths (reuse your existing layout) ---
SCHEMAS_DIR = Path("/mnt/govquery-volume/schemas")
T5_DIR  = Path("/mnt/govquery-volume/models/t5_lora")
SQL_DIR = Path("/mnt/govquery-volume/models/sqlcoder_lora_fp")


# --- Cached model handles ---
_T5_TOK = _T5 = _SQL_TOK = _SQL = None

# --- Utilities ---
def _truncate_semicolon(txt: str) -> str:
    return (txt.split(";")[0] + ";") if ";" in txt else txt.strip()

def _load_schema(code: str) -> Dict[str, Any]:
    with open(SCHEMAS_DIR / f"{code}.json") as f:
        return json.load(f)

def _schema_columns(schema_dict: Dict[str, Any]):
    return [c["name"] for c in schema_dict["columns"] if c["name"] not in ("geography_id","year")]

def _extract_select_columns(sql: str):
    m = re.search(r"SELECT\s+(.*?)\s+FROM", sql, flags=re.IGNORECASE|re.DOTALL)
    if not m: return set()
    raw = m.group(1)
    parts = [p.strip() for p in raw.split(",") if p.strip()]
    cols = []
    for p in parts:
        p = re.sub(r"\b(CAST|SUM|COUNT|AVG|MIN|MAX)\s*\((.*?)\)", r"\2", p, flags=re.IGNORECASE)
        p = re.sub(r"\bas\b.*$", "", p, flags=re.IGNORECASE).strip().strip('`"')
        if p == "*": 
            continue
        if "." in p: 
            p = p.split(".")[-1].strip('`"')
        cols.append(p)
    return set(cols)

# --- Prompt builder ---
# If your notebook already defines build_schema_aware_prompt(...), we'll use it.
def _prompt(schema_dict: Dict[str, Any], question: str) -> str:
    if "build_schema_aware_prompt" in globals() and callable(globals()["build_schema_aware_prompt"]):
        return globals()["build_schema_aware_prompt"](schema_dict, question)
    # Minimal fallback
    code = schema_dict.get("table_code","UNKNOWN")
    name = schema_dict.get("table_name","")
    lines = [f"- {c['name']} ({c['type']}): {c.get('description','')}" for c in schema_dict["columns"]][:200]
    cols = "\n".join(lines)
    return f"### Table: {code} ({name})\nColumns:\n{cols}\n\nQuestion: {question}\nSQL:"

def _guardrails_prompt(schema_dict: Dict[str, Any], question: str) -> str:
    allowed = ", ".join(_schema_columns(schema_dict)[:200])
    base = _prompt(schema_dict, question)
    rules = (
        "\n\n### Rules\n"
        f"- Use ONLY these columns (case-sensitive): {allowed}.\n"
        "- Do NOT prefix columns with table names (use total_population, not b01001.total_population).\n"
        "- Do NOT use '*'.\n"
        "- Do NOT add aggregates unless the question asks.\n"
        "- End with a single semicolon.\n"
    )
    return base + rules

# --- Load adapters (lazy) ---
def _load_t5():
    global _T5_TOK, _T5
    if _T5 is None:
        _T5_TOK = AutoTokenizer.from_pretrained(str(T5_DIR))
        _T5 = AutoModelForSeq2SeqLM.from_pretrained(str(T5_DIR)).to("cuda" if torch.cuda.is_available() else "cpu")
    return _T5_TOK, _T5

def _load_sql():
    global _SQL_TOK, _SQL
    if _SQL is None:
        _SQL_TOK = AutoTokenizer.from_pretrained(str(SQL_DIR), use_fast=True)
        if _SQL_TOK.pad_token is None:
            _SQL_TOK.pad_token = _SQL_TOK.eos_token
        _SQL_TOK.padding_side = "right"
        _SQL = AutoModelForCausalLM.from_pretrained(str(SQL_DIR), device_map="auto")
        _SQL.config.pad_token_id = _SQL_TOK.pad_token_id
    return _SQL_TOK, _SQL

def _gen_t5(prompt: str) -> str:
    tok, mdl = _load_t5()
    enc = tok(prompt, return_tensors="pt", truncation=True, padding=True, max_length=1536).to(mdl.device)
    out = mdl.generate(**enc, max_new_tokens=256, do_sample=False, num_beams=1)
    return _truncate_semicolon(tok.decode(out[0], skip_special_tokens=True))

def _gen_sqlcoder(prompt: str) -> str:
    tok, mdl = _load_sql()
    enc = tok(prompt, return_tensors="pt", truncation=True, padding=True, max_length=2048).to(mdl.device)
    out = mdl.generate(**enc, max_new_tokens=256, do_sample=False, num_beams=1)
    txt = tok.decode(out[0], skip_special_tokens=True)
    return _truncate_semicolon(txt[len(prompt):].strip())

def _choose_model(prompt_len: int) -> str:
    return "t5" if prompt_len < 1400 else "sqlcoder"

def _guarded_generate(model_name: str, schema_dict: Dict[str, Any], question: str, retries: int = 2):
    # 1) vanilla
    p = _prompt(schema_dict, question)
    sql = _gen_t5(p) if model_name=="t5" else _gen_sqlcoder(p)
    used = _extract_select_columns(sql)
    allowed = set(_schema_columns(schema_dict))
    n = 0
    while (not used.issubset(allowed)) and n < retries:
        bad = ", ".join(sorted(used - allowed)) if used else "unknown"
        p2 = _guardrails_prompt(schema_dict, question) + f"\n(Note: previous attempt used invalid columns: {bad})\nSQL:"
        sql = _gen_t5(p2) if model_name=="t5" else _gen_sqlcoder(p2)
        used = _extract_select_columns(sql)
        n += 1
    return sql, {"retries": n, "ok": used.issubset(allowed), "used": sorted(used)}

# =========================
# Novice: intent → slots
# =========================

# 1) Measure lexicon (start small; expand from logs)
_MEASURE_LEXICON = [
    # (regex, table_code, column, description)
    (r"\b(unemployed|unemployment)\b",        "B23025", "unemployed", "Unemployed, age 16+"),
    (r"\b(employed)\b",                        "B23025", "employed", "Employed, age 16+"),
    (r"\b(in labor force|labor force)\b",     "B23025", "in_labor_force", "In labor force, age 16+"),
    (r"\b(total population)\b",               "B01001", "total_population", "Total population"),
    (r"\b(asian)\b",                          "B02001", "asian_alone", "Asian alone"),
    (r"\b(black|african[- ]american)\b",      "B02001", "black_or_african_american_alone", "Black or African American alone"),
    (r"\b(median household income)\b",        "B19013", "median_household_income_in_the_past_12_months_in_2022_inflation_adjusted_dollars", "Median household income"),
]

_STATE_NAME_TO_FIPS = {
    "alabama": "04000US01", "alaska": "04000US02", "arizona": "04000US04", "arkansas": "04000US05",
    "california": "04000US06", "colorado": "04000US08", "connecticut": "04000US09", "delaware": "04000US10",
    "district of columbia": "04000US11", "florida": "04000US12", "georgia": "04000US13", "hawaii": "04000US15",
    "idaho": "04000US16", "illinois": "04000US17", "indiana": "04000US18", "iowa": "04000US19",
    "kansas": "04000US20", "kentucky": "04000US21", "louisiana": "04000US22", "maine": "04000US23",
    "maryland": "04000US24", "massachusetts": "04000US25", "michigan": "04000US26", "minnesota": "04000US27",
    "mississippi": "04000US28", "missouri": "04000US29", "montana": "04000US30", "nebraska": "04000US31",
    "nevada": "04000US32", "new hampshire": "04000US33", "new jersey": "04000US34", "new mexico": "04000US35",
    "new york": "04000US36", "north carolina": "04000US37", "north dakota": "04000US38", "ohio": "04000US39",
    "oklahoma": "04000US40", "oregon": "04000US41", "pennsylvania": "04000US42", "rhode island": "04000US44",
    "south carolina": "04000US45", "south dakota": "04000US46", "tennessee": "04000US47", "texas": "04000US48",
    "utah": "04000US49", "vermont": "04000US50", "virginia": "04000US51", "washington": "04000US53",
    "west virginia": "04000US54", "wisconsin": "04000US55", "wyoming": "04000US56",
    "puerto rico": "04000US72",
}

# USPS abbreviation map (UPPERCASE keys only)
_USPS_ABBR_TO_FIPS = {
    "AL":"04000US01","AK":"04000US02","AZ":"04000US04","AR":"04000US05","CA":"04000US06","CO":"04000US08",
    "CT":"04000US09","DE":"04000US10","DC":"04000US11","FL":"04000US12","GA":"04000US13","HI":"04000US15",
    "ID":"04000US16","IL":"04000US17","IN":"04000US18","IA":"04000US19","KS":"04000US20","KY":"04000US21",
    "LA":"04000US22","ME":"04000US23","MD":"04000US24","MA":"04000US25","MI":"04000US26","MN":"04000US27",
    "MS":"04000US28","MO":"04000US29","MT":"04000US30","NE":"04000US31","NV":"04000US32","NH":"04000US33",
    "NJ":"04000US34","NM":"04000US35","NY":"04000US36","NC":"04000US37","ND":"04000US38","OH":"04000US39",
    "OK":"04000US40","OR":"04000US41","PA":"04000US42","RI":"04000US44","SC":"04000US45","SD":"04000US46",
    "TN":"04000US47","TX":"04000US48","UT":"04000US49","VT":"04000US50","VA":"04000US51","WA":"04000US53",
    "WV":"04000US54","WI":"04000US55","WY":"04000US56","PR":"04000US72",
}


def _latest_year_from_schemas(default_fallback=2022) -> int:
    years = []
    for p in SCHEMAS_DIR.glob("*.json"):
        try:
            s = json.load(open(p))
            years.append(int(s.get("example_row", {}).get("year", default_fallback)))
        except Exception:
            pass
    return max(years) if years else default_fallback

def detect_measure(question: str) -> Optional[Tuple[str,str,str]]:
    q = question.lower()
    for pat, table_code, col, desc in _MEASURE_LEXICON:
        if re.search(pat, q):
            return table_code, col, desc
    return None

def detect_year(question: str, fallback: Optional[int]=None) -> int:
    q = question
    m = re.search(r"\b(20\d{2})\b", q)
    if m:
        return int(m.group(1))
    return fallback if fallback is not None else _latest_year_from_schemas(2022)

def detect_geography(question: str):
    """
    Returns (level, geography_id, label) with safe abbreviation handling:
    - Prefer full state names (case-insensitive)
    - Abbreviations must be ALL-CAPS two-letter tokens and appear with light context
      to avoid collisions (e.g., 'in IN', ', OR', 'for TX', 'TX state').
    """
    q = question
    q_lc = q.lower()

    # Nation
    if any(k in q_lc for k in ("united states", "u.s.", "us")):
        return "nation", "01000US", "United States"

    # Full state name (strict contains with word boundaries)
    padded = f" {q_lc} "
    for name, gid in _STATE_NAME_TO_FIPS.items():
        if f" {name} " in padded:
            return "state", gid, name.title()

    # Abbreviation detection (ALL-CAPS tokens only) + light context
    import re
    tokens = re.findall(r"[A-Za-z]+", q)  # words only
    for i, tok in enumerate(tokens):
        if len(tok) == 2 and tok.isupper() and tok in _USPS_ABBR_TO_FIPS:
            prev = tokens[i-1].lower() if i > 0 else ""
            nxt  = tokens[i+1].lower() if i+1 < len(tokens) else ""
            if prev in {"in","of","for","from","by","across","in,", "in."} or nxt in {"state","states"}:
                gid = _USPS_ABBR_TO_FIPS[tok]
                return "state", gid, tok
            # also allow punctuation context like ", TX"
            if i > 0 and tokens[i-1].endswith(","):
                gid = _USPS_ABBR_TO_FIPS[tok]
                return "state", gid, tok

    # Optional: "by county in <STATE/ABBR>"
    m = re.search(r"\bby county in ([A-Za-z][A-Za-z\s]+)\b", q, flags=re.IGNORECASE)
    if m:
        name = m.group(1).strip().lower()
        if name in _STATE_NAME_TO_FIPS:
            st = _STATE_NAME_TO_FIPS[name][-2:]
            return "county_all_in_state", f"05000US{st}%", name.title()
        if name.isupper() and name in _USPS_ABBR_TO_FIPS:
            st = _USPS_ABBR_TO_FIPS[name][-2:]
            return "county_all_in_state", f"05000US{st}%", name

    # Default to nation; Novice mode will show this assumption
    return "unknown", "01000US", "United States"

def novice_plan(question: str) -> Dict[str, Any]:
    """
    Turns a vague question into a concrete plan with visible assumptions.
    """
    m = detect_measure(question)
    y = detect_year(question)
    level, gid, geo_label = detect_geography(question)

    plan = {
        "question": question,
        "assumptions": {},
        "intent": {},
        "warnings": [],
    }

    # Measure
    if m:
        table_code, column, desc = m
        plan["intent"].update({"table_code": table_code, "column": column, "measure_desc": desc})
    else:
        plan["warnings"].append("Measure ambiguous; default mapping may not exist.")
        plan["intent"].update({"table_code": None, "column": None, "measure_desc": None})

    # Year
    plan["intent"]["year"] = y
    plan["assumptions"]["year"] = y

    # Geography
    plan["intent"]["geography_id"] = gid
    plan["assumptions"]["geography"] = {"level": level, "label": geo_label, "geography_id": gid}
    if level == "unknown":
        plan["warnings"].append("Geography not specified; defaulted to United States.")

    # Breakdown signal
    plan["intent"]["breakdown"] = "by_county" if "by county" in question.lower() else None

    return plan

def enforce_geo(sql: str, geo_id: str, year: int) -> str:
    """
    Replace/insert the exact geography_id and year filters so the final SQL
    matches the plan (prevents models from copying example_row geography).
    """
    s = sql.strip().rstrip(";")

    # Normalize whitespace
    s = re.sub(r"\s+", " ", s)

    # If there's already a geography_id filter, replace it with the exact one.
    if re.search(r"(?i)\bgeography_id\s*=\s*'[^']+'", s):
        s = re.sub(r"(?i)\bgeography_id\s*=\s*'[^']+'", f"geography_id = '{geo_id}'", s)
    else:
        # If there's a WHERE, append; else create WHERE
        if re.search(r"(?i)\bwhere\b", s):
            s += f" AND geography_id = '{geo_id}'"
        else:
            s += f" WHERE geography_id = '{geo_id}'"

    # Ensure year filter exists (replace or append)
    if re.search(r"(?i)\byear\s*=\s*\d{4}\b", s):
        s = re.sub(r"(?i)\byear\s*=\s*\d{4}\b", f"year = {year}", s)
    else:
        if re.search(r"(?i)\bwhere\b", s):
            # If WHERE exists but we just added geography, year may still be missing
            if not re.search(r"(?i)\byear\s*=\s*\d{4}\b", s):
                s += f" AND year = {year}"
        else:
            s += f" WHERE year = {year}"

    return s.strip() + ";"


# =========================
# Novice → SQL via models (schema-guarded)
# =========================
def novice_generate_sql(question: str) -> Dict[str, Any]:
    plan = novice_plan(question)

    # If we couldn't map measure → table/column, stop early with a helpful note.
    if not plan["intent"]["table_code"] or not plan["intent"]["column"]:
        return {
            "ok": False,
            "message": "I couldn't confidently map your measure to a Census/ACS column. Try rephrasing (e.g., 'unemployed', 'total population', 'Asian').",
            "plan": plan
        }

    table_code = plan["intent"]["table_code"]
    column     = plan["intent"]["column"]
    year       = plan["intent"]["year"]
    geo_id     = plan["intent"]["geography_id"]
    breakdown  = plan["intent"]["breakdown"]

    schema = _load_schema(table_code)

    # Compose a clarified question the models can nail (we still show the user's original in 'plan')
    if breakdown == "by_county" and geo_id.startswith("05000US"):
        clarified_q = (f"Return {column.replace('_',' ')} for geography_id = '{geo_id}' in {year}. " "Use exactly this geography_id value and do not infer geography from any examples.")
    elif breakdown == "by_county" and geo_id == "05000US48%":
        clarified_q = f"List {column.replace('_',' ')} by county in Texas in {year}."
    else:
        clarified_q = f"Return {column.replace('_',' ')} for this geography in {year}."

    ptext = _prompt(schema, clarified_q)
    model_name = _choose_model(len(ptext))

    # Guarded generation
    sql, meta = _guarded_generate(model_name, schema, clarified_q, retries=2)
    sql = enforce_geo(sql, geo_id, year)   # <-- add this line


    # (Optional) If breakdown by county + state pattern, ensure WHERE clause matches our intent
    if breakdown == "by_county" and geo_id == "05000US48%":
        # If the model produced a single-row WHERE, you can force a LIKE filter for counties inside TX.
        if "LIKE '05000US48%'" not in sql and "05000US48" not in sql:
            sql = f"SELECT geography_id, {column}\nFROM {schema['sql_ddl'].split()[2]}\nWHERE geography_id LIKE '05000US48%' AND year = {year};"

    out = {
        "ok": True,
        "model": model_name,
        "sql": sql,
        "meta": meta,
        "plan": plan
    }

    # Be polite with VRAM in notebooks
    gc.collect(); torch.cuda.empty_cache()
    return out

# =========================
# Example calls (you can comment these out)
# =========================
# ex1 = novice_generate_sql("How many people were unemployed in 2022?")          # defaults to US if geo missing
# ex2 = novice_generate_sql("How many people were unemployed in Texas in 2022?")
# ex3 = novice_generate_sql("How many Asian people by county in Texas in 2022?")
# ex1, ex2, ex3

In [22]:
# ---- Safe local SQLCoder loader (bypasses library _load_sql) ----
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, gc

_SQL_SAFE_TOK = None
_SQL_SAFE = None

def _safe_sql_loader(SQL_DIR):
    """
    Load SQLCoder on a single device (CPU or CUDA), without accelerate hooks.
    This avoids the 'meta device' error from device_map='auto'.
    """
    global _SQL_SAFE_TOK, _SQL_SAFE
    if _SQL_SAFE is None:
        _SQL_SAFE_TOK = AutoTokenizer.from_pretrained(str(SQL_DIR), use_fast=True)
        if _SQL_SAFE_TOK.pad_token is None:
            _SQL_SAFE_TOK.pad_token = _SQL_SAFE_TOK.eos_token
        _SQL_SAFE_TOK.padding_side = "right"

        device = "cuda" if torch.cuda.is_available() else "cpu"
        dtype  = torch.bfloat16 if device == "cuda" else torch.float32

        _SQL_SAFE = AutoModelForCausalLM.from_pretrained(str(SQL_DIR), torch_dtype=dtype)
        _SQL_SAFE.to(device)
        _SQL_SAFE.config.pad_token_id = _SQL_SAFE_TOK.pad_token_id

    return _SQL_SAFE_TOK, _SQL_SAFE

# ---- Override _gen_sqlcoder to use the safe loader ----
def _gen_sqlcoder(prompt: str) -> str:
    tok, mdl = _safe_sql_loader(SQL_DIR)   # <-- uses our safe loader, not the library _load_sql
    enc = tok(prompt, return_tensors="pt", truncation=True, padding=True, max_length=2048).to(mdl.device)
    out = mdl.generate(**enc, max_new_tokens=256, do_sample=False, num_beams=1)
    txt = tok.decode(out[0], skip_special_tokens=True)
    # Trim the echoed prompt if necessary (common with causal LMs)
    return (txt[len(prompt):].strip() if txt.startswith(prompt) else txt).split(";")[0].strip() + ";"

# optional: clear any half-loaded models
gc.collect(); torch.cuda.empty_cache()


# Test Prompts

In [23]:
# ============================================
# Prompt → SQL smoke tester (notebook-only)
# - Change QUESTION / TABLE / MODEL below
# - Prints the exact schema-aware prompt + model SQL
# - Uses your existing load_schema, build_schema_aware_prompt, query
# - If TABLE="auto", uses detect_measure() (from Novice Mode) to pick the table
# ============================================

# --- knobs: edit these three values to test ---
QUESTION = "How many people were unemployed in 2022?"
TABLE    = "auto"      # "auto" or explicit like "B23025"
MODEL    = "auto"      # "auto" | "t5" | "sqlcoder"

# ---- wrapper (no re-implementation of core functions) ----
import json

# Minimal alias so the tester cell can call `query(...)`
def query(table_code: str, question: str, force_model: str | None = None):
    schema = load_schema(table_code)
    prompt = build_schema_aware_prompt(schema, question)
    model = force_model or ("t5" if len(prompt) < 1400 else "sqlcoder")
    sql, meta = _guarded_generate(model, schema, question, retries=2)
    return {"model": model, "sql": sql, "meta": meta, "table": table_code, "question": question}

def prompt2sql_test(question: str, table: str = "auto", model: str = "auto"):
    # 1) resolve table code
    if table == "auto":
        if "detect_measure" in globals() and callable(globals()["detect_measure"]):
            det = detect_measure(question)  # expected: (table_code, column, desc)
            if not det:
                raise ValueError("Auto-select couldn't infer a table from the question. Set TABLE explicitly (e.g., 'B23025').")
            table_code = det[0]
            print(f"[auto] Selected table: {table_code}")
        else:
            raise ValueError("detect_measure() not found in this notebook. Run Novice Mode cell or set TABLE explicitly.")
    else:
        table_code = table

    # 2) build the exact schema-aware prompt (this is the string fed to the model initially)
    schema = load_schema(table_code)
    prompt = build_schema_aware_prompt(schema, question)

    # 3) generate SQL via your existing guarded path
    result = query(table_code, question) if model == "auto" else query(table_code, question, force_model=model)

    # 4) pretty print
    print("\n=== Schema-Aware Prompt Sent to Model ===\n")
    print(prompt)

    print("\n=== Generated SQL ===\n")
    print(result.get("sql", "").strip())

    print("\n=== Meta ===")
    meta = {
        "table": result.get("table", table_code),
        "question": result.get("question", question),
        "model_used": result.get("model", model),
        "guard_retries": result.get("meta", {}).get("retries"),
        "guard_ok": result.get("meta", {}).get("ok"),
    }
    print(json.dumps(meta, indent=2))
    return {"prompt": prompt, **result}

# ---- run it with the knobs set above ----
_ = prompt2sql_test(QUESTION, TABLE, MODEL)


[auto] Selected table: B23025


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



=== Schema-Aware Prompt Sent to Model ===

{'table_code': 'B23025', 'table_name': 'Employment Status for the Population 16 Years and Over', 'geography_levels': ['tract', 'county', 'state'], 'primary_keys': ['geography_id', 'year'], 'columns': [{'name': 'geography_id', 'type': 'VARCHAR', 'description': 'Geographic identifier'}, {'name': 'year', 'type': 'INTEGER', 'description': 'ACS 5‑year end year'}, {'name': 'total_population_16_over', 'type': 'INTEGER', 'description': 'Population 16\xa0years and over'}, {'name': 'labor_force', 'type': 'INTEGER', 'description': 'In labor force'}, {'name': 'civilian_labor_force', 'type': 'INTEGER', 'description': 'In civilian labor force'}, {'name': 'employed', 'type': 'INTEGER', 'description': 'Employed population'}, {'name': 'unemployed', 'type': 'INTEGER', 'description': 'Unemployed population'}, {'name': 'armed_forces', 'type': 'INTEGER', 'description': 'In armed forces'}, {'name': 'not_in_labor_force', 'type': 'INTEGER', 'description': 'Not in la

In [24]:
res = novice_generate_sql("How many people were unemployed in 2022?")
print(res["sql"])
print(res["plan"])   # shows the assumptions (e.g., default geo/year)


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


SELECT b23025.unemployed FROM b23025 WHERE b23025.geography_id = '01000US' AND b23025.year = 2022;
{'question': 'How many people were unemployed in 2022?', 'assumptions': {'year': 2022, 'geography': {'level': 'unknown', 'label': 'United States', 'geography_id': '01000US'}}, 'intent': {'table_code': 'B23025', 'column': 'unemployed', 'measure_desc': 'Unemployed, age 16+', 'year': 2022, 'geography_id': '01000US', 'breakdown': None}, 'warnings': ['Geography not specified; defaulted to United States.']}
